# TESS Light Curve — Exhaustive Feature Engineering
**ISRO Hackathon | science_sector1.parquet**

This notebook converts each TESS light curve (time, flux, flux_err arrays) into a single,
rich feature vector. Every section is self-contained and the notebook runs top-to-bottom
without manual intervention.

---

## 1. Library Installation

In [ ]:
# ── Core scientific stack ────────────────────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "numpy", "pandas", "scipy", "astropy",
    "PyWavelets", "statsmodels", "tqdm",
    "pyarrow",           # parquet I/O
    "antropy",           # entropy: sample, approximate, permutation
    "nolds",             # nonlinear dynamics (Hurst, DFA, Lyapunov)
]

OPTIONAL = [
    "tsfresh",           # comprehensive time-series features
    "catch22",           # 22 canonical time-series features
]

def _install(pkg):
    print(f"  Installing {pkg} ...", end=" ")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("OK")
        return True
    else:
        print(f"FAILED — {result.stderr.strip()[-120:]}")
        return False

print("=== Installing required packages ===")
for p in PACKAGES:
    _install(p)

print("\n=== Installing optional packages ===")
AVAILABLE_OPTIONAL = set()
for p in OPTIONAL:
    ok = _install(p)
    if ok:
        AVAILABLE_OPTIONAL.add(p.lower())

print(f"\nOptional available: {AVAILABLE_OPTIONAL}")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.signal as ss
import scipy.stats as stats
import scipy.fft as fft
import pywt
from tqdm.auto import tqdm
from astropy.timeseries import LombScargle
from statsmodels.tsa.stattools import acf, pacf
import gc, os, time as _time, traceback, json
from pathlib import Path

# ── Optional imports ─────────────────────────────────────────────────────────
try:
    import antropy as ant
    HAS_ANTROPY = True
except ImportError:
    HAS_ANTROPY = False
    print("antropy not available — entropy features will use fallback implementations")

try:
    import nolds
    HAS_NOLDS = True
except ImportError:
    HAS_NOLDS = False
    print("nolds not available — nonlinear dynamics features skipped")

try:
    import tsfresh
    from tsfresh.feature_extraction import MinimalFCParameters, EfficientFCParameters
    HAS_TSFRESH = True
except ImportError:
    HAS_TSFRESH = False

try:
    import catch22
    HAS_CATCH22 = True
except ImportError:
    HAS_CATCH22 = False

print(f"\nLibraries loaded.")
print(f"  antropy : {HAS_ANTROPY}")
print(f"  nolds   : {HAS_NOLDS}")
print(f"  tsfresh : {HAS_TSFRESH}")
print(f"  catch22 : {HAS_CATCH22}")

## 2. Dataset Loading

In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
DATA_PATH = "science_sector1.parquet"   # adjust path if running locally

print(f"Loading {DATA_PATH} ...")
df_raw = pd.read_parquet(DATA_PATH)
print("Done.")

# Basic inspection
print("\n── Shape ──────────────────────────────────────────")
print(f"  Rows : {df_raw.shape[0]:,}")
print(f"  Cols : {df_raw.shape[1]}")

print("\n── Column dtypes ──────────────────────────────────")
print(df_raw.dtypes)

print("\n── Memory usage ───────────────────────────────────")
mem_mb = df_raw.memory_usage(deep=True).sum() / 1e6
print(f"  {mem_mb:.1f} MB")

print("\n── Scalar columns — missing values ────────────────")
scalar_cols = ["target_id", "sector", "mission", "label"]
print(df_raw[scalar_cols].isnull().sum())

print("\n── Scalar columns — basic stats ───────────────────")
print(df_raw[scalar_cols].describe(include="all"))

print("\n── Label distribution ─────────────────────────────")
print(df_raw["label"].value_counts())

print("\n── Array length stats ─────────────────────────────")
for col in ["time", "flux", "flux_err"]:
    lens = df_raw[col].apply(len)
    print(f"  {col:8s}: min={lens.min():,}  max={lens.max():,}  mean={lens.mean():,.1f}  std={lens.std():,.1f}")

print("\n── Duplicate target_ids ───────────────────────────")
print(f"  {df_raw['target_id'].duplicated().sum()} duplicates")

print("\n── Sample row ─────────────────────────────────────")
row0 = df_raw.iloc[0]
print(f"  target_id : {row0['target_id']}")
print(f"  time      : [{row0['time'][0]:.4f} ... {row0['time'][-1]:.4f}]  ({len(row0['time'])} pts)")
print(f"  flux      : [{row0['flux'].min():.4f} ... {row0['flux'].max():.4f}]")
print(f"  flux_err  : [{row0['flux_err'].min():.6f} ... {row0['flux_err'].max():.6f}]")

### 2b. Exploratory Data Analysis — Visual Inspection

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2b. EDA — Visual Inspection of Raw Light Curves
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import matplotlib
matplotlib.use("Agg")  # Colab uses inline backend; this fallback prevents errors
import matplotlib.pyplot as plt

def plot_sample_light_curves(df_raw, n_samples=6, seed=42):
    """Plot a grid of randomly-sampled raw light curves."""
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(df_raw), size=min(n_samples, len(df_raw)), replace=False)

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    axes = axes.flatten()

    for i, row_idx in enumerate(idxs):
        row = df_raw.iloc[row_idx]
        t, f, e = row["time"], row["flux"], row["flux_err"]
        ax = axes[i]
        ax.errorbar(t, f, yerr=e, fmt=".k", ms=0.5, alpha=0.3, elinewidth=0.2)
        ax.set_title(f"target_id={row['target_id']}  (n={len(t):,})", fontsize=9)
        ax.set_xlabel("Time (BTJD)", fontsize=7)
        ax.set_ylabel("Flux", fontsize=7)
        ax.tick_params(labelsize=7)

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Sample TESS Light Curves — Sector 1", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig("sample_lightcurves.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: sample_lightcurves.png")

plot_sample_light_curves(df_raw)

# ── Distribution of array lengths ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lens = df_raw["time"].apply(len)
axes[0].hist(lens, bins=40, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Number of observations per light curve"); axes[0].set_ylabel("Count")
axes[0].set_title("Light curve length distribution")

cadences = df_raw["time"].apply(lambda t: float(np.median(np.diff(np.sort(t)))) * 24 * 60)
axes[1].hist(cadences, bins=40, color="darkorange", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("Median cadence (minutes)"); axes[1].set_ylabel("Count")
axes[1].set_title("Median cadence distribution")

snrs = df_raw.apply(
    lambda r: float(np.std(r["flux"])) / float(np.mean(r["flux_err"])+1e-30), axis=1
)
axes[2].hist(np.clip(snrs, 0, 200), bins=40, color="seagreen", edgecolor="white", linewidth=0.5)
axes[2].set_xlabel("SNR (std_flux / mean_flux_err)"); axes[2].set_ylabel("Count")
axes[2].set_title("SNR distribution (clipped at 200)")

plt.suptitle("Dataset Characteristics", fontsize=13)
plt.tight_layout()
plt.savefig("dataset_overview.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: dataset_overview.png")


## 3. Data Validation

In [ ]:
# ── Validation report ─────────────────────────────────────────────────────────
issues = []
fixes  = []

print("Running data validation ...\n")

# ── 3a. Duplicate target IDs ──────────────────────────────────────────────────
n_dup_ids = df_raw["target_id"].duplicated().sum()
if n_dup_ids:
    issues.append(f"Duplicate target_ids: {n_dup_ids}")
else:
    print("[OK] No duplicate target_ids")

# ── 3b. Inconsistent array lengths (time vs flux vs flux_err) ─────────────────
bad_len_mask = df_raw.apply(
    lambda r: (len(r["time"]) != len(r["flux"])) or
              (len(r["time"]) != len(r["flux_err"])), axis=1
)
n_bad_len = bad_len_mask.sum()
if n_bad_len:
    issues.append(f"{n_bad_len} rows have inconsistent array lengths")
else:
    print("[OK] All array lengths consistent")

# ── 3c. NaN / Inf in arrays ───────────────────────────────────────────────────
n_nan_flux = df_raw["flux"].apply(lambda a: np.any(~np.isfinite(a))).sum()
n_nan_time = df_raw["time"].apply(lambda a: np.any(~np.isfinite(a))).sum()
n_nan_err  = df_raw["flux_err"].apply(lambda a: np.any(~np.isfinite(a))).sum()
if n_nan_flux or n_nan_time or n_nan_err:
    issues.append(f"Rows with non-finite values — time:{n_nan_time}, flux:{n_nan_flux}, flux_err:{n_nan_err}")
else:
    print("[OK] No NaN/Inf in array values")

# ── 3d. Negative flux_err ─────────────────────────────────────────────────────
n_neg_err = df_raw["flux_err"].apply(lambda a: np.any(a < 0)).sum()
if n_neg_err:
    issues.append(f"{n_neg_err} rows contain negative flux_err values")
    # Safe fix: take absolute value
    df_raw["flux_err"] = df_raw["flux_err"].apply(lambda a: np.abs(a))
    fixes.append("Applied abs() to flux_err arrays")
else:
    print("[OK] No negative flux_err values")

# ── 3e. Observation gaps > 2× median cadence ─────────────────────────────────
def _count_gaps(t_arr, gap_factor=2.0):
    if len(t_arr) < 2:
        return 0
    dt = np.diff(np.sort(t_arr))
    dt = dt[dt > 0]
    if len(dt) == 0:
        return 0
    med = np.median(dt)
    return int(np.sum(dt > gap_factor * med))

gap_counts = df_raw["time"].apply(_count_gaps)
n_gapped = (gap_counts > 0).sum()
print(f"[INFO] {n_gapped} light curves have observation gaps > 2× median cadence")

# ── 3f. Zero-variance flux (flat / dead channels) ─────────────────────────────
n_flat = df_raw["flux"].apply(lambda a: np.std(a) == 0).sum()
if n_flat:
    issues.append(f"{n_flat} light curves have zero flux variance (flat)")
else:
    print("[OK] No zero-variance (flat) flux curves")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n── Validation Summary ─────────────────────────────")
if issues:
    print("Issues found:")
    for i in issues: print(f"  ⚠ {i}")
else:
    print("  No critical issues found.")
if fixes:
    print("Fixes applied:")
    for f in fixes: print(f"  ✓ {f}")

# Keep working copy
df = df_raw.copy()
print(f"\nWorking dataset: {len(df)} rows")

## 4. Feature Engineering

All extractors are pure functions → `extract_all_features(time, flux, flux_err)` returns a flat dict.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HELPER UTILITIES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _safe(func, *args, default=np.nan, **kwargs):
    """Evaluate func(*args) and return default on any exception."""
    try:
        result = func(*args, **kwargs)
        return result if np.isfinite(result) else default
    except Exception:
        return default

def _prefix(d, pfx):
    """Return dict with every key prefixed by pfx."""
    return {f"{pfx}{k}": v for k, v in d.items()}

def _ensure_array(x):
    """Convert to float64 numpy array."""
    return np.asarray(x, dtype=np.float64)

# ── Shannon entropy on a continuous signal (binned) ───────────────────────────
def _shannon_entropy_binned(x, bins=50):
    counts, _ = np.histogram(x, bins=bins)
    counts = counts[counts > 0].astype(float)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-12))

# ── Sample entropy (fallback if antropy unavailable) ──────────────────────────
def _sample_entropy_fallback(x, m=2, r_frac=0.2):
    """Approximate sample entropy via template matching."""
    N = len(x)
    if N < 20:
        return np.nan
    r = r_frac * np.std(x, ddof=1)
    if r == 0:
        return np.nan
    def _count_templates(m_):
        count = 0
        for i in range(N - m_):
            template = x[i:i+m_]
            for j in range(i+1, N - m_):
                if np.max(np.abs(template - x[j:j+m_])) < r:
                    count += 1
        return count
    # Subsample for speed on long signals
    sub = x[:500] if N > 500 else x
    Cm  = _count_templates(m)
    Cm1 = _count_templates(m+1)
    if Cm == 0:
        return np.nan
    return -np.log(Cm1 / Cm) if Cm1 > 0 else np.nan

# ── Permutation entropy (fallback) ────────────────────────────────────────────
def _perm_entropy_fallback(x, order=3, delay=1):
    N = len(x)
    if N < order * delay:
        return np.nan
    from itertools import permutations
    from math import factorial
    count = {}
    for i in range(N - (order-1)*delay):
        seq = x[i:i+order*delay:delay]
        key = tuple(np.argsort(seq))
        count[key] = count.get(key, 0) + 1
    total = sum(count.values())
    probs = np.array(list(count.values())) / total
    return -np.sum(probs * np.log2(probs + 1e-12))

print("Utilities defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.1  OBSERVATION FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_observation_features(time):
    """
    Sampling structure of the light curve (independent of flux values).
    """
    t = _ensure_array(time)
    t_sort = np.sort(t)
    dt = np.diff(t_sort)
    dt = dt[dt > 0]   # remove exact duplicates

    n = len(t)
    duration = float(t_sort[-1] - t_sort[0]) if n > 1 else 0.0
    med_cad  = float(np.median(dt)) if len(dt) > 0 else np.nan

    # Gap analysis
    gap_thresh  = 2.0 * med_cad if np.isfinite(med_cad) else np.nan
    if np.isfinite(gap_thresh):
        gaps = dt[dt > gap_thresh]
        n_gaps      = len(gaps)
        max_gap     = float(gaps.max()) if n_gaps else 0.0
        mean_gap    = float(gaps.mean()) if n_gaps else 0.0
        gap_frac    = float(gaps.sum() / duration) if duration > 0 else 0.0
    else:
        n_gaps = max_gap = mean_gap = gap_frac = np.nan

    # Sampling density
    samp_density = n / duration if duration > 0 else np.nan

    return {
        "obs_count"         : n,
        "obs_duration"      : duration,
        "obs_cadence_median": med_cad,
        "obs_cadence_mean"  : float(np.mean(dt)) if len(dt) > 0 else np.nan,
        "obs_cadence_std"   : float(np.std(dt))  if len(dt) > 0 else np.nan,
        "obs_cadence_min"   : float(dt.min())    if len(dt) > 0 else np.nan,
        "obs_cadence_max"   : float(dt.max())    if len(dt) > 0 else np.nan,
        "obs_cadence_iqr"   : float(np.percentile(dt,75)-np.percentile(dt,25)) if len(dt)>0 else np.nan,
        "obs_n_gaps"        : n_gaps,
        "obs_max_gap"       : max_gap,
        "obs_mean_gap"      : mean_gap,
        "obs_gap_fraction"  : gap_frac,
        "obs_sampling_density": samp_density,
        "obs_time_start"    : float(t_sort[0]),
        "obs_time_end"      : float(t_sort[-1]),
        "obs_regularity"    : float(np.std(dt)/np.mean(dt)) if (len(dt)>0 and np.mean(dt)!=0) else np.nan,
    }

print("4.1 Observation features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.2  STATISTICAL FEATURES  (basic + robust + noise)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_statistical_features(flux, flux_err):
    """Standard, robust, and noise statistics of the flux array."""
    f = _ensure_array(flux)
    e = _ensure_array(flux_err)

    # ── Standard statistics ───────────────────────────────────────────────
    mu       = float(np.mean(f))
    med      = float(np.median(f))
    var      = float(np.var(f, ddof=1))
    std      = float(np.std(f, ddof=1))
    rms      = float(np.sqrt(np.mean(f**2)))
    f_min    = float(f.min())
    f_max    = float(f.max())
    f_range  = f_max - f_min
    cv       = std / mu if mu != 0 else np.nan      # coefficient of variation
    skew     = float(stats.skew(f))
    kurt     = float(stats.kurtosis(f))             # Fisher (excess)

    # ── Robust statistics ─────────────────────────────────────────────────
    mad          = float(np.median(np.abs(f - med)))  # MAD
    iqr          = float(np.percentile(f, 75) - np.percentile(f, 25))
    p5, p10, p25, p75, p90, p95 = [
        float(np.percentile(f, p)) for p in [5, 10, 25, 75, 90, 95]
    ]
    trimmed_mean = float(stats.trim_mean(f, 0.1))    # 10 % trimmed mean
    winsorized_std = float(np.std(np.clip(f, p5, p95), ddof=1))

    # ── Normalised flux deviation ─────────────────────────────────────────
    # (median-normalised for transit/eclipse detection)
    f_norm = f / med if med != 0 else f
    norm_std   = float(np.std(f_norm, ddof=1))
    norm_range = float(f_norm.max() - f_norm.min())

    # ── Noise features ────────────────────────────────────────────────────
    e_mean      = float(np.mean(e))
    e_std       = float(np.std(e, ddof=1))
    e_median    = float(np.median(e))
    noise_var   = float(np.mean(e**2))               # mean squared error
    # SNR: median flux amplitude / mean error
    snr         = float(std / e_mean) if e_mean > 0 else np.nan
    # Reduced chi-squared (vs. flat model = mean)
    e_safe      = np.where(e > 0, e, np.median(e[e > 0]) if np.any(e > 0) else 1.0)
    chi2_red    = float(np.mean(((f - mu) / e_safe)**2))

    # ── Higher-order moments ──────────────────────────────────────────────
    hyperskewness = float(stats.moment(f, 5) / (std**5)) if std > 0 else np.nan
    hyperflatness = float(stats.moment(f, 6) / (std**6)) if std > 0 else np.nan

    # ── Waveform shape factors (used in vibration analysis) ───────────────
    abs_f = np.abs(f)
    peak_val     = float(abs_f.max())
    crest_factor = peak_val / rms if rms > 0 else np.nan
    shape_factor = rms / (np.mean(abs_f)) if np.mean(abs_f) > 0 else np.nan
    impulse_factor = peak_val / np.mean(abs_f) if np.mean(abs_f) > 0 else np.nan
    clearance_factor = peak_val / (np.mean(np.sqrt(abs_f))**2) if np.mean(np.sqrt(abs_f)) > 0 else np.nan

    return {
        # Standard
        "stat_mean"           : mu,
        "stat_median"         : med,
        "stat_variance"       : var,
        "stat_std"            : std,
        "stat_rms"            : rms,
        "stat_min"            : f_min,
        "stat_max"            : f_max,
        "stat_range"          : f_range,
        "stat_cv"             : cv,
        "stat_skewness"       : skew,
        "stat_kurtosis"       : kurt,
        "stat_hyperskewness"  : hyperskewness,
        "stat_hyperflatness"  : hyperflatness,
        # Robust
        "robust_mad"          : mad,
        "robust_iqr"          : iqr,
        "robust_p5"           : p5,
        "robust_p10"          : p10,
        "robust_p25"          : p25,
        "robust_p75"          : p75,
        "robust_p90"          : p90,
        "robust_p95"          : p95,
        "robust_trimmed_mean" : trimmed_mean,
        "robust_winsorized_std": winsorized_std,
        # Normalised
        "norm_std"            : norm_std,
        "norm_range"          : norm_range,
        # Noise
        "noise_snr"           : snr,
        "noise_err_mean"      : e_mean,
        "noise_err_std"       : e_std,
        "noise_err_median"    : e_median,
        "noise_variance"      : noise_var,
        "noise_chi2_reduced"  : chi2_red,
        # Shape factors
        "shape_crest_factor"  : crest_factor,
        "shape_shape_factor"  : shape_factor,
        "shape_impulse_factor": impulse_factor,
        "shape_clearance_factor": clearance_factor,
    }

print("4.2 Statistical features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.3  TREND FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_trend_features(time, flux):
    """Linear and polynomial trend estimation; residual statistics."""
    t = _ensure_array(time)
    f = _ensure_array(flux)
    t_norm = (t - t.mean()) / (t.std() + 1e-12)   # normalise for stability

    # ── Linear fit ────────────────────────────────────────────────────────
    try:
        lin_coeffs = np.polyfit(t_norm, f, 1)
        lin_slope     = float(lin_coeffs[0])
        lin_intercept = float(lin_coeffs[1])
        lin_pred      = np.polyval(lin_coeffs, t_norm)
        lin_resid     = f - lin_pred
        ss_res  = float(np.sum(lin_resid**2))
        ss_tot  = float(np.sum((f - f.mean())**2))
        lin_r2  = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
        lin_resid_std  = float(np.std(lin_resid, ddof=2))
        lin_resid_skew = float(stats.skew(lin_resid))
    except Exception:
        lin_slope = lin_intercept = lin_r2 = lin_resid_std = lin_resid_skew = np.nan

    # ── Quadratic (polynomial degree 2) ───────────────────────────────────
    try:
        quad_coeffs = np.polyfit(t_norm, f, 2)
        quad_a = float(quad_coeffs[0])   # curvature
        quad_b = float(quad_coeffs[1])   # linear slope
        quad_pred  = np.polyval(quad_coeffs, t_norm)
        quad_resid = f - quad_pred
        ss_res_q = float(np.sum(quad_resid**2))
        quad_r2  = 1 - ss_res_q/ss_tot if ss_tot > 0 else np.nan
        quad_resid_std = float(np.std(quad_resid, ddof=3))
    except Exception:
        quad_a = quad_b = quad_r2 = quad_resid_std = np.nan

    # ── Residual improvement from linear → quadratic ──────────────────────
    try:
        r2_improvement = float(quad_r2 - lin_r2)
    except Exception:
        r2_improvement = np.nan

    # ── Spearman trend (monotonicity) ─────────────────────────────────────
    try:
        spearman_r, _ = stats.spearmanr(t_norm, f)
        spearman_r = float(spearman_r)
    except Exception:
        spearman_r = np.nan

    # ── Mean flux first half vs second half (activity shift) ──────────────
    n  = len(f)
    h1 = float(np.mean(f[:n//2]))
    h2 = float(np.mean(f[n//2:]))
    half_diff = h2 - h1

    return {
        "trend_lin_slope"       : lin_slope,
        "trend_lin_intercept"   : lin_intercept,
        "trend_lin_r2"          : lin_r2,
        "trend_lin_resid_std"   : lin_resid_std,
        "trend_lin_resid_skew"  : lin_resid_skew,
        "trend_quad_curvature"  : quad_a,
        "trend_quad_slope"      : quad_b,
        "trend_quad_r2"         : quad_r2,
        "trend_quad_resid_std"  : quad_resid_std,
        "trend_r2_improvement"  : r2_improvement,
        "trend_spearman_r"      : spearman_r,
        "trend_half_diff"       : half_diff,
        "trend_half1_mean"      : h1,
        "trend_half2_mean"      : h2,
    }

print("4.3 Trend features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.4  DERIVATIVE FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_derivative_features(time, flux):
    """First- and second-order time derivatives of flux."""
    t = _ensure_array(time)
    f = _ensure_array(flux)

    # Sort by time
    idx = np.argsort(t)
    t, f = t[idx], f[idx]

    dt = np.diff(t)
    dt = np.where(dt == 0, 1e-12, dt)  # avoid division by zero

    # First derivative
    d1 = np.diff(f) / dt

    # Second derivative
    dt2 = 0.5 * (dt[:-1] + dt[1:])  # midpoint spacing for 2nd derivative
    dt2 = np.where(dt2 == 0, 1e-12, dt2)
    d2 = np.diff(d1) / dt2

    def _stats(arr, tag):
        if len(arr) == 0:
            return {f"{tag}mean":np.nan, f"{tag}std":np.nan,
                    f"{tag}max":np.nan, f"{tag}rms":np.nan,
                    f"{tag}skew":np.nan, f"{tag}kurt":np.nan,
                    f"{tag}mad":np.nan}
        return {
            f"{tag}mean" : float(np.mean(arr)),
            f"{tag}std"  : float(np.std(arr, ddof=1)),
            f"{tag}max"  : float(np.max(np.abs(arr))),
            f"{tag}rms"  : float(np.sqrt(np.mean(arr**2))),
            f"{tag}skew" : float(stats.skew(arr)),
            f"{tag}kurt" : float(stats.kurtosis(arr)),
            f"{tag}mad"  : float(np.median(np.abs(arr - np.median(arr)))),
        }

    feat = {}
    feat.update(_stats(d1, "deriv1_"))
    feat.update(_stats(d2, "deriv2_"))

    # Zero-crossings of first derivative (local extrema count)
    sign_changes = np.sum(np.diff(np.sign(d1)) != 0)
    feat["deriv_zero_crossings_d1"] = int(sign_changes)
    feat["deriv_zero_crossings_d2"] = int(np.sum(np.diff(np.sign(d2)) != 0))

    return feat

print("4.4 Derivative features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.5  PEAK / VALLEY FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_peak_features(time, flux):
    """Peak/valley detection using scipy.signal.find_peaks."""
    t = _ensure_array(time)
    f = _ensure_array(flux)

    idx = np.argsort(t)
    t, f = t[idx], f[idx]

    # ── Peaks ─────────────────────────────────────────────────────────────
    peak_idx, peak_props = ss.find_peaks(
        f,
        prominence=np.std(f)*0.3,
        width=3
    )
    n_peaks = len(peak_idx)

    # ── Valleys (invert signal) ────────────────────────────────────────────
    valley_idx, valley_props = ss.find_peaks(
        -f,
        prominence=np.std(f)*0.3,
        width=3
    )
    n_valleys = len(valley_idx)

    def _peak_stats(idx_, values_, props_, tag):
        if len(idx_) == 0:
            return {
                f"{tag}count"          : 0,
                f"{tag}density"        : 0.0,
                f"{tag}mean_height"    : np.nan,
                f"{tag}max_height"     : np.nan,
                f"{tag}mean_prominence": np.nan,
                f"{tag}max_prominence" : np.nan,
                f"{tag}mean_width"     : np.nan,
                f"{tag}mean_spacing"   : np.nan,
                f"{tag}std_spacing"    : np.nan,
                f"{tag}energy"         : np.nan,
            }

        heights = np.abs(values_[idx_])
        proms   = props_.get("prominences", np.full(len(idx_), np.nan))
        widths  = props_.get("widths",      np.full(len(idx_), np.nan))
        times_  = t[idx_]
        spacing = np.diff(times_) if len(times_) > 1 else np.array([np.nan])
        duration_ = t[-1] - t[0] if len(t) > 1 else 1.0

        return {
            f"{tag}count"          : len(idx_),
            f"{tag}density"        : len(idx_) / duration_ if duration_ > 0 else np.nan,
            f"{tag}mean_height"    : float(np.mean(heights)),
            f"{tag}max_height"     : float(np.max(heights)),
            f"{tag}mean_prominence": float(np.mean(proms))   if not np.all(np.isnan(proms)) else np.nan,
            f"{tag}max_prominence" : float(np.max(proms))    if not np.all(np.isnan(proms)) else np.nan,
            f"{tag}mean_width"     : float(np.mean(widths))  if not np.all(np.isnan(widths)) else np.nan,
            f"{tag}mean_spacing"   : float(np.mean(spacing)),
            f"{tag}std_spacing"    : float(np.std(spacing))  if len(spacing) > 1 else np.nan,
            f"{tag}energy"         : float(np.sum(heights**2)),
        }

    feat = {}
    feat.update(_peak_stats(peak_idx,   f,  peak_props,   "peak_"))
    feat.update(_peak_stats(valley_idx, -f, valley_props, "valley_"))

    # Total extrema density
    dur = t[-1] - t[0] if len(t) > 1 else 1.0
    feat["peak_valley_total_count"]   = n_peaks + n_valleys
    feat["peak_valley_total_density"] = (n_peaks + n_valleys) / dur if dur > 0 else np.nan
    feat["peak_valley_ratio"]         = n_peaks / n_valleys if n_valleys > 0 else np.nan

    return feat

print("4.5 Peak/valley features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.6  FFT / SPECTRAL FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_fft_features(time, flux, n_top=10):
    """Spectral features via FFT on uniformly sampled residuals."""
    t = _ensure_array(time)
    f = _ensure_array(flux)
    idx = np.argsort(t)
    t, f = t[idx], f[idx]

    # Subtract linear trend (detrend) before FFT
    f_detrend = ss.detrend(f, type="linear")

    N  = len(f_detrend)
    dt = float(np.median(np.diff(t)))
    if dt <= 0:
        dt = 1.0

    # FFT
    F       = fft.rfft(f_detrend * np.hanning(N))
    freqs   = fft.rfftfreq(N, d=dt)
    power   = np.abs(F)**2
    power_n = power / (power.sum() + 1e-30)  # normalised for entropy

    # Dominant frequency
    dom_idx   = int(np.argmax(power[1:]) + 1)  # skip DC
    dom_freq  = float(freqs[dom_idx])
    dom_power = float(power[dom_idx])
    dom_period = 1.0 / dom_freq if dom_freq > 0 else np.nan

    # Top-N frequencies
    sorted_idx = np.argsort(power[1:])[::-1] + 1
    top_n = min(n_top, len(sorted_idx))
    top_freqs  = freqs[sorted_idx[:top_n]]
    top_powers = power[sorted_idx[:top_n]]

    feat = {
        "fft_dom_freq"    : dom_freq,
        "fft_dom_period"  : dom_period,
        "fft_dom_power"   : dom_power,
        "fft_dom_power_frac": dom_power / (power.sum() + 1e-30),
    }

    for i in range(top_n):
        feat[f"fft_top{i+1}_freq"]  = float(top_freqs[i])
        feat[f"fft_top{i+1}_power"] = float(top_powers[i])

    # ── Spectral descriptors ──────────────────────────────────────────────
    # Spectral entropy
    pn = power_n[power_n > 0]
    feat["fft_spectral_entropy"] = float(-np.sum(pn * np.log(pn + 1e-30)))

    # Spectral centroid (weighted mean frequency)
    feat["fft_spectral_centroid"] = float(
        np.sum(freqs * power) / (np.sum(power) + 1e-30)
    )

    # Spectral bandwidth (weighted std)
    centroid = feat["fft_spectral_centroid"]
    feat["fft_spectral_bandwidth"] = float(
        np.sqrt(np.sum(((freqs - centroid)**2) * power) / (np.sum(power) + 1e-30))
    )

    # Spectral rolloff (85% of energy)
    cumpower = np.cumsum(power)
    rolloff_thresh = 0.85 * cumpower[-1]
    rolloff_idx = np.searchsorted(cumpower, rolloff_thresh)
    feat["fft_spectral_rolloff"] = float(freqs[min(rolloff_idx, len(freqs)-1)])

    # Spectral flatness (geometric mean / arithmetic mean of power)
    p_pos = power[power > 0]
    if len(p_pos) > 0:
        feat["fft_spectral_flatness"] = float(
            np.exp(np.mean(np.log(p_pos + 1e-30))) / (np.mean(p_pos) + 1e-30)
        )
    else:
        feat["fft_spectral_flatness"] = np.nan

    # Power ratio: low / high frequency (split at Nyquist/4)
    nyq = freqs[-1]
    split = nyq / 4.0
    low_power  = float(power[freqs <= split].sum())
    high_power = float(power[freqs >  split].sum())
    feat["fft_low_high_power_ratio"] = low_power / (high_power + 1e-30)

    # DC offset (zeroth FFT component = mean)
    feat["fft_dc_power"] = float(power[0])

    return feat

print("4.6 FFT features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.7  AUTOCORRELATION FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_autocorrelation_features(flux, n_lags=50):
    """ACF, PACF, decorrelation lag, and periodicity strength."""
    f = _ensure_array(flux)

    # Subsample if very long (ACF is O(N*lags))
    max_pts = 5000
    f_sub = f[::max(1, len(f)//max_pts)]

    n_lags_use = min(n_lags, len(f_sub)//2 - 1)

    feat = {}

    # ── ACF ───────────────────────────────────────────────────────────────
    try:
        acf_vals = acf(f_sub, nlags=n_lags_use, fft=True)
        for i in [1, 2, 5, 10, 20, 50]:
            if i <= n_lags_use:
                feat[f"acf_lag{i}"] = float(acf_vals[i])

        # Decorrelation lag: first lag where |ACF| < 1/e
        thresh = 1.0 / np.e
        below  = np.where(np.abs(acf_vals[1:]) < thresh)[0]
        feat["acf_decorrelation_lag"] = int(below[0] + 1) if len(below) > 0 else n_lags_use

        # ACF sum (persistence measure)
        feat["acf_sum_abs"]   = float(np.sum(np.abs(acf_vals[1:])))
        feat["acf_max_abs"]   = float(np.max(np.abs(acf_vals[1:])))

        # First peak of ACF (rough period)
        acf_peaks, _ = ss.find_peaks(acf_vals[1:])
        if len(acf_peaks) > 0:
            feat["acf_first_peak_lag"]   = int(acf_peaks[0] + 1)
            feat["acf_first_peak_value"] = float(acf_vals[acf_peaks[0]+1])
        else:
            feat["acf_first_peak_lag"]   = np.nan
            feat["acf_first_peak_value"] = np.nan

    except Exception:
        for k in ["acf_decorrelation_lag","acf_sum_abs","acf_max_abs",
                  "acf_first_peak_lag","acf_first_peak_value"]:
            feat[k] = np.nan

    # ── PACF ──────────────────────────────────────────────────────────────
    try:
        n_pacf = min(20, len(f_sub)//2 - 1)
        pacf_vals = pacf(f_sub, nlags=n_pacf)
        for i in [1, 2, 5, 10]:
            if i <= n_pacf:
                feat[f"pacf_lag{i}"] = float(pacf_vals[i])
        feat["pacf_max_abs"] = float(np.max(np.abs(pacf_vals[1:])))
    except Exception:
        for k in ["pacf_lag1","pacf_lag2","pacf_lag5","pacf_lag10","pacf_max_abs"]:
            feat[k] = np.nan

    return feat

print("4.7 Autocorrelation features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.8  ENTROPY FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_entropy_features(flux):
    """Shannon, sample, approximate, and permutation entropy."""
    f = _ensure_array(flux)

    # Subsample for expensive computations
    f_short = f[::max(1, len(f)//1000)]   # ≤1000 points for SampEn / ApEn

    feat = {}

    # ── Shannon entropy ────────────────────────────────────────────────────
    feat["ent_shannon"] = _safe(_shannon_entropy_binned, f)

    # ── Sample entropy ─────────────────────────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_sample"] = float(ant.sample_entropy(f_short))
        except Exception:
            feat["ent_sample"] = np.nan
    else:
        feat["ent_sample"] = _sample_entropy_fallback(f_short)

    # ── Approximate entropy ────────────────────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_approximate"] = float(ant.app_entropy(f_short))
        except Exception:
            feat["ent_approximate"] = np.nan
    else:
        feat["ent_approximate"] = np.nan   # expensive fallback skipped

    # ── Permutation entropy ────────────────────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_permutation"] = float(ant.perm_entropy(f_short, order=3, normalize=True))
        except Exception:
            feat["ent_permutation"] = np.nan
    else:
        feat["ent_permutation"] = _perm_entropy_fallback(f_short, order=3)

    # ── Spectral entropy ───────────────────────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_spectral"] = float(ant.spectral_entropy(
                f, sf=1.0, method="welch", normalize=True
            ))
        except Exception:
            feat["ent_spectral"] = np.nan
    else:
        feat["ent_spectral"] = np.nan

    # ── SVD entropy ───────────────────────────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_svd"] = float(ant.svd_entropy(f_short, order=3, normalize=True))
        except Exception:
            feat["ent_svd"] = np.nan
    else:
        feat["ent_svd"] = np.nan

    # ── Lempel-Ziv complexity (binarised) ─────────────────────────────────
    if HAS_ANTROPY:
        try:
            feat["ent_lziv"] = float(ant.lziv_complexity(f_short > np.median(f_short), normalize=True))
        except Exception:
            feat["ent_lziv"] = np.nan
    else:
        feat["ent_lziv"] = np.nan

    return feat

print("4.8 Entropy features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.9  HJORTH PARAMETERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_hjorth_features(flux):
    """
    Hjorth Activity, Mobility, and Complexity —
    originally from EEG analysis, useful for any non-stationary signal.
    """
    f  = _ensure_array(flux)
    d1 = np.diff(f)
    d2 = np.diff(d1)

    var_f  = float(np.var(f,  ddof=1))
    var_d1 = float(np.var(d1, ddof=1))
    var_d2 = float(np.var(d2, ddof=1))

    activity   = var_f
    mobility   = np.sqrt(var_d1 / var_f) if var_f > 0 else np.nan
    mob_d1     = np.sqrt(var_d2 / var_d1) if var_d1 > 0 else np.nan
    complexity = mob_d1 / mobility if mobility and mobility > 0 else np.nan

    return {
        "hjorth_activity"  : activity,
        "hjorth_mobility"  : float(mobility)  if np.isfinite(mobility)  else np.nan,
        "hjorth_complexity": float(complexity) if np.isfinite(complexity) else np.nan,
    }

print("4.9 Hjorth features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.10  WAVELET FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_wavelet_features(flux, wavelet="db4", levels=5):
    """
    Multi-level discrete wavelet decomposition.
    Extracts statistics of approximation and detail coefficients at each level.
    """
    f = _ensure_array(flux)

    feat = {}

    try:
        max_level = pywt.dwt_max_level(len(f), pywt.Wavelet(wavelet))
        levels_use = min(levels, max_level)
        coeffs = pywt.wavedec(f, wavelet, level=levels_use)
        # coeffs[0] = approximation at deepest level
        # coeffs[1..] = detail at levels from deepest to finest

        for lvl, c in enumerate(coeffs):
            tag = f"wav_approx_" if lvl == 0 else f"wav_detail{lvl}_"
            if lvl > 0:
                tag = f"wav_d{lvl}_"
            else:
                tag = "wav_approx_"

            c = np.asarray(c, dtype=np.float64)
            feat[f"{tag}mean"]    = float(np.mean(c))
            feat[f"{tag}std"]     = float(np.std(c,  ddof=1)) if len(c)>1 else np.nan
            feat[f"{tag}energy"]  = float(np.sum(c**2))
            feat[f"{tag}entropy"] = float(_shannon_entropy_binned(c, bins=min(30, len(c))))
            feat[f"{tag}max_abs"] = float(np.max(np.abs(c)))
            feat[f"{tag}skew"]    = float(stats.skew(c)) if len(c) > 2 else np.nan
            feat[f"{tag}kurt"]    = float(stats.kurtosis(c)) if len(c) > 3 else np.nan

        # Energy ratio: fraction of total energy at each level
        total_energy = sum(np.sum(c**2) for c in coeffs) + 1e-30
        for lvl, c in enumerate(coeffs):
            c = np.asarray(c, dtype=np.float64)
            tag = "wav_approx" if lvl == 0 else f"wav_d{lvl}"
            feat[f"{tag}_energy_ratio"] = float(np.sum(c**2) / total_energy)

    except Exception as e:
        feat["wav_error"] = str(e)

    return feat

print("4.10 Wavelet features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.11  LOMB–SCARGLE FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_lombscargle_features(time, flux, n_top=5):
    """
    Lomb-Scargle periodogram for unevenly sampled data.
    Extracts dominant period, top-N periods, power, and peak ratios.
    """
    t = _ensure_array(time)
    f = _ensure_array(flux)

    feat = {}

    try:
        # Subsample for speed (LS is O(N log N) with NFFT approach)
        max_pts = 5000
        if len(t) > max_pts:
            idx = np.linspace(0, len(t)-1, max_pts, dtype=int)
            t, f = t[idx], f[idx]

        # Period grid
        dt_med   = float(np.median(np.abs(np.diff(t))))
        duration = float(t.max() - t.min())
        min_freq = 1.0 / duration   if duration > 0 else 1e-6
        max_freq = 1.0 / (2*dt_med) if dt_med   > 0 else 10.0

        if min_freq >= max_freq:
            raise ValueError("Invalid frequency range")

        ls   = LombScargle(t, f)
        freq = np.linspace(min_freq, max_freq, 5000)
        power = ls.power(freq)

        periods = 1.0 / freq

        # Best period
        best_idx  = int(np.argmax(power))
        best_freq  = float(freq[best_idx])
        best_power = float(power[best_idx])
        best_period = float(periods[best_idx])

        feat["ls_best_freq"]   = best_freq
        feat["ls_best_period"] = best_period
        feat["ls_best_power"]  = best_power

        # Top-N peaks
        peak_idxs, _ = ss.find_peaks(power)
        if len(peak_idxs) == 0:
            peak_idxs = np.argsort(power)[::-1][:n_top]
        else:
            peak_idxs = peak_idxs[np.argsort(power[peak_idxs])[::-1][:n_top]]

        for i, pi in enumerate(peak_idxs[:n_top]):
            feat[f"ls_top{i+1}_period"] = float(periods[pi])
            feat[f"ls_top{i+1}_power"]  = float(power[pi])

        # Power statistics
        feat["ls_power_mean"]   = float(np.mean(power))
        feat["ls_power_std"]    = float(np.std(power))
        feat["ls_power_max"]    = float(np.max(power))

        # Peak-to-second-peak ratio (periodicity strength)
        sorted_powers = np.sort(power)[::-1]
        feat["ls_peak_ratio_1_2"] = (
            float(sorted_powers[0] / sorted_powers[1])
            if len(sorted_powers) > 1 and sorted_powers[1] > 0 else np.nan
        )

        # False alarm probability at best power
        try:
            fap = ls.false_alarm_probability(best_power)
            feat["ls_fap"] = float(fap)
        except Exception:
            feat["ls_fap"] = np.nan

    except Exception as exc:
        for key in ["ls_best_freq","ls_best_period","ls_best_power",
                    "ls_power_mean","ls_power_std","ls_power_max",
                    "ls_peak_ratio_1_2","ls_fap"]:
            feat[key] = np.nan

    return feat

print("4.11 Lomb-Scargle features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.12  TRANSIT / DIP FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_transit_features(time, flux, sigma_thresh=3.0):
    """
    Transit / dip detection: looks for flux drops below baseline.
    Uses a sigma-clipping approach on a median-normalised light curve.
    """
    t = _ensure_array(time)
    f = _ensure_array(flux)

    idx = np.argsort(t)
    t, f = t[idx], f[idx]

    feat = {}

    try:
        # Median-normalise
        med = np.median(f)
        if med == 0:
            med = 1.0
        fn = f / med

        std_n = np.std(fn, ddof=1)
        thresh = 1.0 - sigma_thresh * std_n   # dip threshold

        # Identify dip regions
        dip_mask = fn < thresh
        dip_changes = np.diff(dip_mask.astype(int))
        starts = np.where(dip_changes ==  1)[0] + 1
        ends   = np.where(dip_changes == -1)[0] + 1

        # Handle edge cases
        if dip_mask[0]:
            starts = np.concatenate([[0], starts])
        if dip_mask[-1]:
            ends = np.concatenate([ends, [len(fn)]])

        n_dips = min(len(starts), len(ends))
        feat["transit_n_dips"] = n_dips

        if n_dips == 0:
            for k in ["transit_deepest_dip","transit_mean_dip_depth",
                      "transit_mean_dip_duration","transit_dip_depth_std",
                      "transit_ingress_slope","transit_egress_slope",
                      "transit_symmetry","transit_dip_spacing",
                      "transit_dip_energy","transit_period_estimate"]:
                feat[k] = np.nan
        else:
            dip_depths    = []
            dip_durations = []
            ingress_slopes  = []
            egress_slopes   = []
            dip_centers     = []

            for s, e in zip(starts[:n_dips], ends[:n_dips]):
                seg = fn[s:e]
                depth = float(1.0 - seg.min())
                dip_depths.append(depth)

                t_seg = t[s:e]
                dur   = float(t_seg[-1] - t_seg[0]) if len(t_seg) > 1 else 0.0
                dip_durations.append(dur)
                dip_centers.append(float(np.mean(t_seg)))

                # Ingress/egress slopes
                if s > 0 and e < len(fn) - 1:
                    # Ingress: slope from one point before dip to dip minimum
                    min_idx = s + int(np.argmin(seg))
                    dt_in  = t[min_idx] - t[s-1]
                    dt_out = t[e]      - t[min_idx]
                    if dt_in > 0:
                        ingress_slopes.append((fn[min_idx]-fn[s-1]) / dt_in)
                    if dt_out > 0:
                        egress_slopes.append((fn[e]-fn[min_idx]) / dt_out)

            feat["transit_deepest_dip"]       = float(max(dip_depths))
            feat["transit_mean_dip_depth"]     = float(np.mean(dip_depths))
            feat["transit_dip_depth_std"]      = float(np.std(dip_depths)) if n_dips > 1 else np.nan
            feat["transit_mean_dip_duration"]  = float(np.mean(dip_durations))
            feat["transit_dip_energy"]         = float(np.sum(np.array(dip_depths)**2))

            feat["transit_ingress_slope"] = (
                float(np.mean(ingress_slopes)) if ingress_slopes else np.nan
            )
            feat["transit_egress_slope"] = (
                float(np.mean(egress_slopes)) if egress_slopes else np.nan
            )

            # Symmetry: |ingress| / |egress| close to 1 = symmetric transit
            if ingress_slopes and egress_slopes:
                mean_in  = abs(np.mean(ingress_slopes))
                mean_out = abs(np.mean(egress_slopes))
                denom    = max(mean_in, mean_out) + 1e-30
                feat["transit_symmetry"] = float(min(mean_in, mean_out) / denom)
            else:
                feat["transit_symmetry"] = np.nan

            # Dip spacing (estimated period for repeating transits)
            if len(dip_centers) > 1:
                spacings = np.diff(dip_centers)
                feat["transit_dip_spacing"]      = float(np.mean(spacings))
                feat["transit_period_estimate"]  = float(np.median(spacings))
            else:
                feat["transit_dip_spacing"]     = np.nan
                feat["transit_period_estimate"] = np.nan

    except Exception:
        for k in ["transit_n_dips","transit_deepest_dip","transit_mean_dip_depth",
                  "transit_mean_dip_duration","transit_dip_depth_std",
                  "transit_ingress_slope","transit_egress_slope",
                  "transit_symmetry","transit_dip_spacing",
                  "transit_dip_energy","transit_period_estimate"]:
            feat[k] = np.nan

    return feat

print("4.12 Transit/dip features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.13  ROLLING WINDOW FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_rolling_features(flux, windows=(16, 32, 64, 128)):
    """
    Rolling-window statistics across multiple window sizes.
    Captures local variability structure not visible in global statistics.
    """
    f = _ensure_array(flux)
    series = pd.Series(f)

    feat = {}

    for w in windows:
        if w >= len(f):
            continue
        roll = series.rolling(window=w, center=True)

        r_mean = roll.mean().dropna().values
        r_std  = roll.std().dropna().values
        r_var  = roll.var().dropna().values
        r_max  = roll.max().dropna().values
        r_min  = roll.min().dropna().values

        # Energy in each window
        r_energy = (series**2).rolling(window=w, center=True).mean().dropna().values

        tag = f"roll{w}_"

        # Summary statistics OF the rolling statistic
        feat[f"{tag}mean_mean"]   = float(np.mean(r_mean))
        feat[f"{tag}std_mean"]    = float(np.mean(r_std))
        feat[f"{tag}std_std"]     = float(np.std(r_std))  if len(r_std) > 1 else np.nan
        feat[f"{tag}std_max"]     = float(np.max(r_std))  if len(r_std) > 0 else np.nan
        feat[f"{tag}var_mean"]    = float(np.mean(r_var))
        feat[f"{tag}var_std"]     = float(np.std(r_var))  if len(r_var) > 1 else np.nan
        feat[f"{tag}max_mean"]    = float(np.mean(r_max))
        feat[f"{tag}min_mean"]    = float(np.mean(r_min))
        feat[f"{tag}range_mean"]  = float(np.mean(r_max - r_min))
        feat[f"{tag}energy_mean"] = float(np.mean(r_energy))
        feat[f"{tag}energy_std"]  = float(np.std(r_energy)) if len(r_energy) > 1 else np.nan

        # Stationarity proxy: coefficient of variation of rolling std
        if np.mean(r_std) != 0:
            feat[f"{tag}cv_std"] = float(np.std(r_std) / np.mean(r_std))
        else:
            feat[f"{tag}cv_std"] = np.nan

    return feat

print("4.13 Rolling window features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.14  NONLINEAR DYNAMICS FEATURES  (Hurst, DFA, Lyapunov)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_nonlinear_features(flux):
    """
    Hurst exponent, DFA (detrended fluctuation analysis), and
    largest Lyapunov exponent via nolds (if available).
    """
    f = _ensure_array(flux)
    f_short = f[::max(1, len(f)//2000)]   # ≤2000 pts for speed

    feat = {}

    if not HAS_NOLDS:
        for k in ["nonlin_hurst","nonlin_dfa","nonlin_lyapunov","nonlin_corr_dim"]:
            feat[k] = np.nan
        return feat

    try:
        feat["nonlin_hurst"] = float(nolds.hurst_rs(f_short))
    except Exception:
        feat["nonlin_hurst"] = np.nan

    try:
        feat["nonlin_dfa"] = float(nolds.dfa(f_short))
    except Exception:
        feat["nonlin_dfa"] = np.nan

    try:
        feat["nonlin_lyapunov"] = float(nolds.lyap_r(f_short, emb_dim=3))
    except Exception:
        feat["nonlin_lyapunov"] = np.nan

    try:
        feat["nonlin_corr_dim"] = float(nolds.corr_dim(f_short, emb_dim=2))
    except Exception:
        feat["nonlin_corr_dim"] = np.nan

    return feat

print("4.14 Nonlinear dynamics features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.15  CATCH22 FEATURES  (canonical time-series)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_catch22_features(flux):
    """22 canonical time-series features (Lubba et al. 2019)."""
    if not HAS_CATCH22:
        return {}

    f = _ensure_array(flux)
    f_sub = f[::max(1, len(f)//2000)].tolist()

    feat = {}
    try:
        result = catch22.catch22_all(f_sub)
        for name, val in zip(result["names"], result["values"]):
            feat[f"c22_{name}"] = float(val) if np.isfinite(val) else np.nan
    except Exception:
        pass

    return feat

print("4.15 catch22 features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.16  FLUX-ERROR CROSS FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_flux_error_cross_features(flux, flux_err):
    """
    Relationships between flux and flux_err.
    These can reveal systematic noise, detector saturation, or astrophysical
    variability correlated with brightness.
    """
    f = _ensure_array(flux)
    e = _ensure_array(flux_err)

    feat = {}

    try:
        # Pearson correlation between flux and flux_err
        r, p = stats.pearsonr(f, e)
        feat["cross_flux_err_pearsonr"] = float(r)
        feat["cross_flux_err_pearsonp"] = float(p)
    except Exception:
        feat["cross_flux_err_pearsonr"] = np.nan
        feat["cross_flux_err_pearsonp"] = np.nan

    try:
        # Spearman correlation
        r_sp, _ = stats.spearmanr(f, e)
        feat["cross_flux_err_spearmanr"] = float(r_sp)
    except Exception:
        feat["cross_flux_err_spearmanr"] = np.nan

    # Error-weighted mean flux
    e_safe = np.where(e > 0, e, 1e-10)
    w = 1.0 / e_safe**2
    feat["cross_weighted_mean_flux"]  = float(np.sum(w * f) / np.sum(w))
    feat["cross_weighted_std_flux"]   = float(np.sqrt(1.0 / np.sum(w)))

    # Outlier fraction: flux points > 3σ from error-weighted mean
    wm = feat["cross_weighted_mean_flux"]
    ws = feat["cross_weighted_std_flux"]
    outlier_frac = float(np.mean(np.abs(f - wm) > 3.0 * ws))
    feat["cross_outlier_frac_3sigma"] = outlier_frac
    feat["cross_outlier_frac_5sigma"] = float(np.mean(np.abs(f - wm) > 5.0 * ws))

    # Mean normalised error (proxy for measurement quality)
    mean_norm_err = float(np.mean(e / np.abs(f + 1e-30)))
    feat["cross_mean_norm_err"] = mean_norm_err

    return feat

print("4.16 Flux-error cross features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.17  PHASE-FOLDING FEATURES  (variability structure across phases)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_phase_folding_features(time, flux, ls_period=None, n_bins=20):
    """
    Fold the light curve on the best Lomb-Scargle period and extract
    statistics of the phase-binned light curve.
    Phase-folded features capture periodic modulations (pulsations, eclipses).
    """
    feat = {}

    try:
        if ls_period is None or not np.isfinite(ls_period) or ls_period <= 0:
            for k in ["phase_binned_std","phase_binned_range",
                      "phase_binned_skew","phase_binned_kurt",
                      "phase_min_bin","phase_max_bin",
                      "phase_depth_ratio"]:
                feat[k] = np.nan
            return feat

        t = _ensure_array(time)
        f = _ensure_array(flux)

        # Phase-fold
        phase = (t % ls_period) / ls_period   # in [0, 1)

        # Bin the phase
        bins = np.linspace(0, 1, n_bins + 1)
        bin_idx = np.digitize(phase, bins) - 1
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)

        bin_means = np.array([
            np.mean(f[bin_idx == b]) if np.sum(bin_idx == b) > 0 else np.nan
            for b in range(n_bins)
        ])

        valid = bin_means[np.isfinite(bin_means)]

        if len(valid) < 4:
            for k in ["phase_binned_std","phase_binned_range",
                      "phase_binned_skew","phase_binned_kurt",
                      "phase_min_bin","phase_max_bin","phase_depth_ratio"]:
                feat[k] = np.nan
        else:
            feat["phase_binned_std"]   = float(np.std(valid, ddof=1))
            feat["phase_binned_range"] = float(valid.max() - valid.min())
            feat["phase_binned_skew"]  = float(stats.skew(valid))
            feat["phase_binned_kurt"]  = float(stats.kurtosis(valid))
            feat["phase_min_bin"]      = float(valid.min())
            feat["phase_max_bin"]      = float(valid.max())
            # Depth ratio: how deep the minimum bin is relative to median
            med_bin = float(np.median(valid))
            feat["phase_depth_ratio"] = (
                float((med_bin - valid.min()) / med_bin)
                if med_bin != 0 else np.nan
            )

    except Exception:
        for k in ["phase_binned_std","phase_binned_range",
                  "phase_binned_skew","phase_binned_kurt",
                  "phase_min_bin","phase_max_bin","phase_depth_ratio"]:
            feat[k] = np.nan

    return feat

print("4.17 Phase-folding features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.18  STELLAR VARIABILITY INDICES  (domain-specific)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_variability_index_features(flux, flux_err):
    """
    Variability indices commonly used in time-domain astronomy:
    - Stetson I & J  (correlated variability)
    - Von Neumann η  (adjacent-point scatter)
    - Welch–Stetson Ib
    - Median absolute deviation flux index
    - RoMS (robust median statistic)
    - Normalized excess variance
    """
    f = _ensure_array(flux)
    e = _ensure_array(flux_err)

    e_safe = np.where(e > 0, e, 1e-10)
    feat   = {}

    # Weighted mean
    w  = 1.0 / e_safe**2
    wm = np.sum(w * f) / np.sum(w)

    n  = len(f)
    residuals = (f - wm) / e_safe   # normalised residuals

    # ── Stetson I ─────────────────────────────────────────────────────────
    # Requires pairs; use consecutive pairs
    if n >= 2:
        P_k = residuals[:-1] * residuals[1:]
        stetson_I = float(np.sum(np.sign(P_k) * np.sqrt(np.abs(P_k))) / n)
    else:
        stetson_I = np.nan

    # ── Stetson J ─────────────────────────────────────────────────────────
    if n >= 2:
        stetson_J = float(
            np.sqrt(1.0/(n*(n-1))) * np.sum(P_k)
        )
    else:
        stetson_J = np.nan

    # ── Von Neumann η ─────────────────────────────────────────────────────
    if n >= 2:
        eta = float(
            np.sum((f[1:] - f[:-1])**2) / ((n-1) * np.var(f, ddof=1) + 1e-30)
        )
    else:
        eta = np.nan

    # ── RoMS: robust median statistic ─────────────────────────────────────
    roms = float(np.median(np.abs(residuals)) / 0.6745)

    # ── Normalized excess variance ─────────────────────────────────────────
    # F_var = (sigma^2 - <err^2>) / <flux>^2
    sigma2    = float(np.var(f, ddof=1))
    mean_err2 = float(np.mean(e_safe**2))
    mean_flux = float(np.mean(f))
    if mean_flux != 0:
        f_var = (sigma2 - mean_err2) / (mean_flux**2)
    else:
        f_var = np.nan

    # ── MAD-based flux variability ────────────────────────────────────────
    med_f = np.median(f)
    mad_f = np.median(np.abs(f - med_f))
    mad_index = float(mad_f / med_f) if med_f != 0 else np.nan

    # ── Inverse von Neumann (high = smooth variation) ─────────────────────
    inv_eta = 1.0 / eta if (eta and eta != 0) else np.nan

    feat.update({
        "var_stetson_I"         : stetson_I,
        "var_stetson_J"         : stetson_J,
        "var_von_neumann_eta"   : eta,
        "var_inv_eta"           : inv_eta,
        "var_roms"              : roms,
        "var_f_var"             : f_var,
        "var_mad_index"         : mad_index,
    })

    return feat

print("4.18 Variability index features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.19  STRUCTURE FUNCTION FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_structure_function_features(time, flux, n_bins=20, max_pts=2000):
    """
    First-order structure function SF(τ) = <(f(t+τ) - f(t))^2>.
    The slope of log SF vs log τ is sensitive to the variability process:
    - Slope ≈ 0: white noise
    - Slope ≈ 1: flicker (1/f) noise
    - Slope ≈ 2: random walk
    """
    t = _ensure_array(time)
    f = _ensure_array(flux)

    feat = {}

    try:
        idx = np.argsort(t)
        t, f = t[idx], f[idx]

        # Subsample
        step = max(1, len(t)//max_pts)
        t_s, f_s = t[::step], f[::step]
        N = len(t_s)

        # Compute pairwise time lags and squared differences
        # Use vectorised upper-triangle for small arrays
        if N > 500:
            N = 500
            t_s, f_s = t_s[:N], f_s[:N]

        i_idx, j_idx = np.triu_indices(N, k=1)
        tau = np.abs(t_s[j_idx] - t_s[i_idx])
        diff2 = (f_s[j_idx] - f_s[i_idx])**2

        # Bin by τ on log scale
        tau_min = tau[tau > 0].min() if np.any(tau > 0) else 1e-6
        tau_max = tau.max()
        bins = np.geomspace(tau_min, tau_max, n_bins + 1)
        bin_idx = np.digitize(tau, bins) - 1
        bin_idx = np.clip(bin_idx, 0, n_bins - 1)

        sf_vals = np.array([
            np.mean(diff2[bin_idx == b]) if np.sum(bin_idx == b) > 0 else np.nan
            for b in range(n_bins)
        ])
        tau_centres = (bins[:-1] * bins[1:])**0.5  # geometric mean centres

        valid = np.isfinite(sf_vals) & (sf_vals > 0) & (tau_centres > 0)

        if np.sum(valid) >= 4:
            log_tau = np.log10(tau_centres[valid])
            log_sf  = np.log10(sf_vals[valid])
            slope, intercept, r, _, _ = stats.linregress(log_tau, log_sf)
            feat["sf_slope"]     = float(slope)
            feat["sf_intercept"] = float(intercept)
            feat["sf_r2"]        = float(r**2)
            feat["sf_max"]       = float(sf_vals[valid].max())
            feat["sf_turnover_tau"] = float(tau_centres[valid][np.argmax(sf_vals[valid])])
        else:
            for k in ["sf_slope","sf_intercept","sf_r2","sf_max","sf_turnover_tau"]:
                feat[k] = np.nan

    except Exception:
        for k in ["sf_slope","sf_intercept","sf_r2","sf_max","sf_turnover_tau"]:
            feat[k] = np.nan

    return feat

print("4.19 Structure function features defined.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.20  FLARE FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_flare_features(time, flux, flux_err, sigma_thresh=3.5):
    """
    Detect positive flux excursions (stellar flares).
    Uses sigma-clipping: flare = flux > median + sigma_thresh * sigma.
    Distinct from transit features which target flux dips.
    """
    t = _ensure_array(time)
    f = _ensure_array(flux)
    e = _ensure_array(flux_err)

    idx = np.argsort(t)
    t, f, e = t[idx], f[idx], e[idx]

    feat = {}
    try:
        med   = np.median(f)
        sigma = np.std(f, ddof=1)
        thresh = med + sigma_thresh * sigma

        flare_mask = f > thresh
        flare_changes = np.diff(flare_mask.astype(int))
        starts = np.where(flare_changes ==  1)[0] + 1
        ends   = np.where(flare_changes == -1)[0] + 1

        if flare_mask[0]:  starts = np.concatenate([[0], starts])
        if flare_mask[-1]: ends   = np.concatenate([ends, [len(f)]])

        n_flares = min(len(starts), len(ends))
        feat["flare_count"] = n_flares

        if n_flares == 0:
            for k in ["flare_total_energy","flare_mean_amplitude",
                      "flare_max_amplitude","flare_mean_duration",
                      "flare_rate","flare_energy_frac"]:
                feat[k] = np.nan
        else:
            amplitudes = []
            durations  = []
            energies   = []
            duration_total = t[-1] - t[0] if len(t) > 1 else 1.0

            for s, e_ in zip(starts[:n_flares], ends[:n_flares]):
                seg  = f[s:e_] - med
                amp  = float(seg.max())
                dur  = float(t[e_-1] - t[s]) if e_ > s else 0.0
                # Trapezoid integration for flare energy
                en   = float(np.trapz(seg, t[s:e_])) if e_ > s else 0.0
                amplitudes.append(amp)
                durations.append(dur)
                energies.append(en)

            total_energy = float(np.sum(energies))
            total_flux   = float(np.sum(f * np.gradient(t)))

            feat["flare_total_energy"]   = total_energy
            feat["flare_mean_amplitude"] = float(np.mean(amplitudes))
            feat["flare_max_amplitude"]  = float(np.max(amplitudes))
            feat["flare_mean_duration"]  = float(np.mean(durations))
            feat["flare_rate"]           = n_flares / duration_total if duration_total > 0 else np.nan
            feat["flare_energy_frac"]    = total_energy / (total_flux + 1e-30)

    except Exception:
        for k in ["flare_count","flare_total_energy","flare_mean_amplitude",
                  "flare_max_amplitude","flare_mean_duration",
                  "flare_rate","flare_energy_frac"]:
            feat[k] = np.nan

    return feat

print("4.20 Flare features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.21  QUASI-PERIODICITY / OSCILLATION ENVELOPE FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_envelope_features(flux):
    """
    Compute the analytic signal via Hilbert transform.
    Extracts amplitude envelope and instantaneous frequency statistics.
    Useful for quasi-periodic or amplitude-modulated signals (e.g., delta Scuti).
    """
    f = _ensure_array(flux)
    feat = {}

    try:
        from scipy.signal import hilbert
        analytic  = hilbert(f - np.mean(f))  # remove DC before Hilbert
        envelope  = np.abs(analytic)
        inst_phase= np.unwrap(np.angle(analytic))
        inst_freq = np.diff(inst_phase) / (2.0 * np.pi)   # cycles per sample

        feat["env_mean"]        = float(np.mean(envelope))
        feat["env_std"]         = float(np.std(envelope, ddof=1))
        feat["env_cv"]          = float(np.std(envelope) / (np.mean(envelope)+1e-30))
        feat["env_max"]         = float(envelope.max())
        feat["env_skew"]        = float(stats.skew(envelope))
        feat["env_kurt"]        = float(stats.kurtosis(envelope))
        # Amplitude modulation depth: (max-min)/(max+min)
        feat["env_am_depth"]    = float(
            (envelope.max() - envelope.min()) /
            (envelope.max() + envelope.min() + 1e-30)
        )
        feat["inst_freq_mean"]  = float(np.mean(np.abs(inst_freq)))
        feat["inst_freq_std"]   = float(np.std(inst_freq))
        feat["inst_freq_max"]   = float(np.max(np.abs(inst_freq)))

    except Exception:
        for k in ["env_mean","env_std","env_cv","env_max","env_skew",
                  "env_kurt","env_am_depth","inst_freq_mean",
                  "inst_freq_std","inst_freq_max"]:
            feat[k] = np.nan

    return feat

print("4.21 Envelope (Hilbert) features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.22  BINNED TIME-SEGMENT FEATURES  (temporal evolution)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_segment_features(time, flux, n_segments=5):
    """
    Divide the light curve into equal-time segments and compute per-segment
    statistics. Captures temporal evolution of variability properties.
    Also computes CDPP (Combined Differential Photometric Precision) proxy.
    """
    t = _ensure_array(time)
    f = _ensure_array(flux)

    idx  = np.argsort(t)
    t, f = t[idx], f[idx]

    feat = {}

    try:
        edges = np.linspace(t[0], t[-1], n_segments + 1)
        seg_means, seg_stds, seg_medians, seg_mads = [], [], [], []

        for k in range(n_segments):
            mask = (t >= edges[k]) & (t < edges[k+1])
            seg  = f[mask]
            if len(seg) < 3:
                seg_means.append(np.nan); seg_stds.append(np.nan)
                seg_medians.append(np.nan); seg_mads.append(np.nan)
                continue
            seg_means.append(float(np.mean(seg)))
            seg_stds.append(float(np.std(seg, ddof=1)))
            seg_medians.append(float(np.median(seg)))
            seg_mads.append(float(np.median(np.abs(seg - np.median(seg)))))

        seg_means   = np.array(seg_means,   dtype=float)
        seg_stds    = np.array(seg_stds,    dtype=float)
        seg_medians = np.array(seg_medians, dtype=float)
        seg_mads    = np.array(seg_mads,    dtype=float)

        # Statistics across segments
        valid_m = seg_means[np.isfinite(seg_means)]
        valid_s = seg_stds[np.isfinite(seg_stds)]

        feat["seg_mean_mean"]   = float(np.mean(valid_m))   if len(valid_m) else np.nan
        feat["seg_mean_std"]    = float(np.std(valid_m))    if len(valid_m)>1 else np.nan
        feat["seg_mean_range"]  = float(np.ptp(valid_m))    if len(valid_m) else np.nan
        feat["seg_std_mean"]    = float(np.mean(valid_s))   if len(valid_s) else np.nan
        feat["seg_std_std"]     = float(np.std(valid_s))    if len(valid_s)>1 else np.nan
        feat["seg_std_range"]   = float(np.ptp(valid_s))    if len(valid_s) else np.nan

        # Non-stationarity: Kruskal-Wallis test across segments
        seg_lists = []
        for k in range(n_segments):
            mask = (t >= edges[k]) & (t < edges[k+1])
            seg  = f[mask]
            if len(seg) >= 3:
                seg_lists.append(seg)

        if len(seg_lists) >= 2:
            try:
                H, p_kw = stats.kruskal(*seg_lists)
                feat["seg_kruskal_H"]  = float(H)
                feat["seg_kruskal_p"]  = float(p_kw)
            except Exception:
                feat["seg_kruskal_H"]  = np.nan
                feat["seg_kruskal_p"]  = np.nan
        else:
            feat["seg_kruskal_H"]  = np.nan
            feat["seg_kruskal_p"]  = np.nan

        # ── CDPP proxy (1-hour binned scatter) ───────────────────────────
        # TESS 2-min cadence: 1 hour = 30 points
        bin_size = 30
        n_bins_cdpp = len(f) // bin_size
        if n_bins_cdpp >= 2:
            bins_cdpp = f[:n_bins_cdpp * bin_size].reshape(n_bins_cdpp, bin_size)
            bin_means_cdpp = bins_cdpp.mean(axis=1)
            cdpp = float(np.std(bin_means_cdpp, ddof=1) * 1e6)  # in ppm
        else:
            cdpp = np.nan
        feat["cdpp_proxy_ppm"] = cdpp

    except Exception:
        for k in ["seg_mean_mean","seg_mean_std","seg_mean_range",
                  "seg_std_mean","seg_std_std","seg_std_range",
                  "seg_kruskal_H","seg_kruskal_p","cdpp_proxy_ppm"]:
            feat[k] = np.nan

    return feat

print("4.22 Segment / CDPP features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.23  TSFRESH FEATURES  (if available)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_tsfresh_features_single(flux, target_id=0):
    """
    Extract tsfresh EfficientFCParameters features from a single flux array.
    Uses a subsample (≤ 2000 pts) for speed.
    Returns a flat dict with prefix 'tsf_'.
    """
    if not HAS_TSFRESH:
        return {}

    try:
        from tsfresh import extract_features
        from tsfresh.feature_extraction import EfficientFCParameters
        from tsfresh.utilities.dataframe_functions import impute

        f = _ensure_array(flux)
        # Subsample
        step  = max(1, len(f) // 2000)
        f_sub = f[::step]

        df_ts = pd.DataFrame({
            "id"   : [int(target_id)] * len(f_sub),
            "time" : np.arange(len(f_sub)),
            "flux" : f_sub,
        })

        # Suppress tsfresh internal logging
        import logging
        logging.getLogger("tsfresh").setLevel(logging.ERROR)

        feat_df = extract_features(
            df_ts,
            column_id="id",
            column_sort="time",
            column_value="flux",
            default_fc_parameters=EfficientFCParameters(),
            disable_progressbar=True,
            show_warnings=False,
            n_jobs=1,
        )

        # Rename columns with prefix
        feat_df.columns = [f"tsf_{c}" for c in feat_df.columns]
        # Return as dict
        row = feat_df.iloc[0].to_dict()
        # Replace inf/nan
        return {k: (float(v) if np.isfinite(v) else np.nan)
                for k, v in row.items()
                if isinstance(v, (int, float, np.floating, np.integer))}

    except Exception:
        return {}

print(f"4.23 tsfresh extractor defined  (available={HAS_TSFRESH}).")
if not HAS_TSFRESH:
    print("  tsfresh not installed — section will produce empty dicts (harmless).")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.24  WEIGHTED POWER SPECTRAL FEATURES  (Welch PSD)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_welch_features(time, flux):
    """
    Welch's method PSD for more robust spectral estimation than plain FFT.
    Averages multiple overlapping periodogram segments → lower variance.
    """
    f = _ensure_array(flux)
    t = _ensure_array(time)

    feat = {}
    try:
        dt = float(np.median(np.abs(np.diff(np.sort(t)))))
        if dt <= 0:
            dt = 1.0
        fs = 1.0 / dt

        # Welch PSD
        freq_w, psd = ss.welch(
            f - np.mean(f),
            fs=fs,
            nperseg=min(512, len(f)//4),
            noverlap=None,
            window="hann",
        )

        # Skip DC
        psd  = psd[1:]
        freq_w = freq_w[1:]

        if len(psd) == 0:
            raise ValueError("Empty PSD")

        psd_norm = psd / (psd.sum() + 1e-30)

        dom_idx = int(np.argmax(psd))
        feat["welch_dom_freq"]    = float(freq_w[dom_idx])
        feat["welch_dom_period"]  = 1.0/freq_w[dom_idx] if freq_w[dom_idx]>0 else np.nan
        feat["welch_dom_power"]   = float(psd[dom_idx])

        # Band powers (split into 4 equal-log bands)
        log_f = np.log10(freq_w + 1e-30)
        bands = np.linspace(log_f.min(), log_f.max(), 5)
        for b in range(4):
            mask = (log_f >= bands[b]) & (log_f < bands[b+1])
            feat[f"welch_band{b+1}_power"] = float(psd[mask].sum()) if mask.sum()>0 else 0.0

        # Spectral slope (log PSD vs log freq linear fit)
        valid = psd > 0
        if valid.sum() >= 4:
            sl, _, r, _, _ = stats.linregress(
                np.log10(freq_w[valid]), np.log10(psd[valid])
            )
            feat["welch_spectral_slope"] = float(sl)
            feat["welch_spectral_slope_r2"] = float(r**2)
        else:
            feat["welch_spectral_slope"]    = np.nan
            feat["welch_spectral_slope_r2"] = np.nan

    except Exception:
        for k in ["welch_dom_freq","welch_dom_period","welch_dom_power",
                  "welch_band1_power","welch_band2_power",
                  "welch_band3_power","welch_band4_power",
                  "welch_spectral_slope","welch_spectral_slope_r2"]:
            feat[k] = np.nan

    return feat

print("4.24 Welch PSD features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.25  BOX LEAST-SQUARES (BLS) PROXY FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_bls_proxy_features(time, flux, n_periods=200):
    """
    Simplified BLS proxy using astropy.timeseries.BoxLeastSquares.
    BLS is the gold-standard algorithm for transit detection in exoplanet science.
    """
    feat = {}
    try:
        from astropy.timeseries import BoxLeastSquares
        from astropy import units as u

        t = _ensure_array(time)
        f = _ensure_array(flux)

        # Normalise flux to unit median
        med = np.median(f)
        fn  = f / med if med != 0 else f

        # Subsample for speed
        step = max(1, len(t) // 3000)
        ts, fs = t[::step], fn[::step]

        dur_total = ts[-1] - ts[0]
        periods   = np.linspace(0.5, min(dur_total/2, 30.0), n_periods)
        durations = np.array([0.02, 0.05, 0.1, 0.2])   # in days

        bls    = BoxLeastSquares(ts * u.day, fs)
        result = bls.power(periods * u.day, durations * u.day,
                           oversample=3, objective="snr")

        best_idx    = int(np.argmax(result.power))
        best_period = float(result.period[best_idx].value)
        best_power  = float(result.power[best_idx])
        best_depth  = float(result.depth[best_idx])
        best_dur    = float(result.duration[best_idx].value)
        best_t0     = float(result.transit_time[best_idx].value)

        feat["bls_best_period"]   = best_period
        feat["bls_best_power"]    = best_power
        feat["bls_best_depth"]    = best_depth
        feat["bls_best_duration"] = best_dur
        feat["bls_best_t0"]       = best_t0

        # SNR of best transit
        stats_bls = bls.compute_stats(
            result.period[best_idx],
            result.duration[best_idx],
            result.transit_time[best_idx]
        )
        feat["bls_snr"]           = float(stats_bls["snr"])
        feat["bls_depth_odd"]     = float(stats_bls.get("depth_odd",  [np.nan])[0])
        feat["bls_depth_even"]    = float(stats_bls.get("depth_even", [np.nan])[0])
        # Even-odd depth difference (non-zero → eclipsing binary, not planet)
        feat["bls_odd_even_diff"] = abs(
            feat["bls_depth_odd"] - feat["bls_depth_even"]
        )

        # Power spectrum shape
        feat["bls_power_max"]     = float(result.power.max())
        feat["bls_power_mean"]    = float(result.power.mean())
        feat["bls_power_std"]     = float(result.power.std())

    except Exception:
        for k in ["bls_best_period","bls_best_power","bls_best_depth",
                  "bls_best_duration","bls_best_t0","bls_snr",
                  "bls_depth_odd","bls_depth_even","bls_odd_even_diff",
                  "bls_power_max","bls_power_mean","bls_power_std"]:
            feat[k] = np.nan

    return feat

print("4.25 BLS proxy features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.26  RECURRENCE / CROSSING FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_crossing_features(flux):
    """
    Level-crossing statistics:
    - Mean-crossing rate (zero-crossings of demeaned signal)
    - Fraction of time above/below various flux thresholds
    - Run-length statistics (consecutive above/below median)
    These capture duty cycles and waveform structure not visible in moments.
    """
    f = _ensure_array(flux)
    feat = {}

    try:
        f_dm = f - np.mean(f)          # demeaned
        f_dn = f - np.median(f)        # median-centred

        # Mean-crossings (zero-crossings of demeaned)
        signs = np.sign(f_dm)
        signs[signs == 0] = 1           # treat exact-zero as positive
        crossings = int(np.sum(np.diff(signs) != 0))
        feat["cross_rate_mean"]     = crossings / len(f)

        # Median-crossings
        signs_med = np.sign(f_dn)
        signs_med[signs_med == 0] = 1
        feat["cross_rate_median"]   = int(np.sum(np.diff(signs_med) != 0)) / len(f)

        # Fraction of time above/below thresholds (percentiles of own distribution)
        for pct in [10, 25, 75, 90]:
            thresh  = float(np.percentile(f, pct))
            feat[f"cross_frac_above_p{pct}"] = float(np.mean(f > thresh))

        # Run-length statistics (above/below median)
        above = (f > np.median(f)).astype(int)
        runs  = []
        count = 1
        for i in range(1, len(above)):
            if above[i] == above[i-1]:
                count += 1
            else:
                runs.append(count)
                count = 1
        runs.append(count)
        runs = np.array(runs)

        feat["cross_run_mean"]   = float(runs.mean())
        feat["cross_run_std"]    = float(runs.std())
        feat["cross_run_max"]    = float(runs.max())
        feat["cross_n_runs"]     = len(runs)

    except Exception:
        for k in ["cross_rate_mean","cross_rate_median",
                  "cross_frac_above_p10","cross_frac_above_p25",
                  "cross_frac_above_p75","cross_frac_above_p90",
                  "cross_run_mean","cross_run_std",
                  "cross_run_max","cross_n_runs"]:
            feat[k] = np.nan

    return feat

print("4.26 Level-crossing / run-length features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.27  FLUX QUANTILE BINNING FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_quantile_bin_features(flux, n_bins=10):
    """
    Divide the flux value range into n quantile bins and compute the fraction
    of observations in each bin. Provides a non-parametric flux distribution
    fingerprint. Also computes Wasserstein distance from a uniform distribution.
    """
    f = _ensure_array(flux)
    feat = {}

    try:
        bin_edges = np.percentile(f, np.linspace(0, 100, n_bins + 1))
        # Bin the flux
        bin_ids   = np.digitize(f, bin_edges[1:-1])  # 0..n_bins-1
        counts    = np.bincount(bin_ids, minlength=n_bins)
        fracs     = counts / len(f)

        for i, frac in enumerate(fracs):
            feat[f"qbin_{i+1:02d}_frac"] = float(frac)

        # Wasserstein distance from uniform distribution
        uniform = np.ones(n_bins) / n_bins
        # Simple L1 distance
        feat["qbin_l1_from_uniform"]   = float(np.sum(np.abs(fracs - uniform)))
        feat["qbin_max_frac"]          = float(fracs.max())
        feat["qbin_min_frac"]          = float(fracs.min())
        feat["qbin_entropy"]           = float(
            -np.sum(fracs[fracs>0] * np.log2(fracs[fracs>0] + 1e-12))
        )

    except Exception:
        for i in range(n_bins):
            feat[f"qbin_{i+1:02d}_frac"] = np.nan
        for k in ["qbin_l1_from_uniform","qbin_max_frac","qbin_min_frac","qbin_entropy"]:
            feat[k] = np.nan

    return feat

print("4.27 Quantile-bin distribution features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4.28  MULTI-SCALE VARIABILITY RATIO FEATURES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_multiscale_variability(flux, scales=(2, 4, 8, 16, 32, 64)):
    """
    Compute RMS variability at each temporal scale by computing
    the RMS of successive differences at stride = scale.
    The ratio of variability at different scales reveals the fractal structure.
    Related to Allan deviation in precision timing.
    """
    f = _ensure_array(flux)
    feat = {}

    rms_values = {}
    for s in scales:
        if s >= len(f):
            rms_values[s] = np.nan
            continue
        diffs = f[s:] - f[:-s]
        rms_values[s] = float(np.sqrt(np.mean(diffs**2)))
        feat[f"msvar_rms_scale{s}"] = rms_values[s]

    # Ratios between consecutive scales
    scale_list = sorted(scales)
    for i in range(len(scale_list) - 1):
        s1, s2 = scale_list[i], scale_list[i+1]
        v1, v2 = rms_values.get(s1, np.nan), rms_values.get(s2, np.nan)
        if v1 and v2 and v1 > 0:
            feat[f"msvar_ratio_{s1}_{s2}"] = float(v2 / v1)
        else:
            feat[f"msvar_ratio_{s1}_{s2}"] = np.nan

    # Log-log slope of RMS vs scale (= Hurst-like exponent, simpler than nolds)
    valid_s  = [s for s in scales if np.isfinite(rms_values.get(s, np.nan)) and rms_values[s]>0]
    if len(valid_s) >= 3:
        log_s   = np.log2(valid_s)
        log_rms = np.log2([rms_values[s] for s in valid_s])
        slope, _, r, _, _ = stats.linregress(log_s, log_rms)
        feat["msvar_slope"]  = float(slope)   # ≈ Hurst (H = slope + 0.5 for fractional Brownian motion)
        feat["msvar_slope_r2"] = float(r**2)
    else:
        feat["msvar_slope"]    = np.nan
        feat["msvar_slope_r2"] = np.nan

    return feat

print("4.28 Multi-scale variability features defined.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MASTER EXTRACTOR  — calls all modules, returns flat dict
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_all_features(target_id, sector, time, flux, flux_err):
    """Run all feature extractors and return one flat feature dict."""
    feat = {
        "target_id" : int(target_id),
        "sector"    : int(sector),
    }

    t = _ensure_array(time)
    f = _ensure_array(flux)
    e = _ensure_array(flux_err)

    # ── 4.1  Observation features ─────────────────────────────────────────
    try: feat.update(extract_observation_features(t))
    except Exception as ex: feat["_err_obs"] = str(ex)

    # ── 4.2  Statistical features ─────────────────────────────────────────
    try: feat.update(extract_statistical_features(f, e))
    except Exception as ex: feat["_err_stat"] = str(ex)

    # ── 4.3  Trend features ───────────────────────────────────────────────
    try: feat.update(extract_trend_features(t, f))
    except Exception as ex: feat["_err_trend"] = str(ex)

    # ── 4.4  Derivative features ──────────────────────────────────────────
    try: feat.update(extract_derivative_features(t, f))
    except Exception as ex: feat["_err_deriv"] = str(ex)

    # ── 4.5  Peak / valley features ───────────────────────────────────────
    try: feat.update(extract_peak_features(t, f))
    except Exception as ex: feat["_err_peak"] = str(ex)

    # ── 4.6  FFT features ─────────────────────────────────────────────────
    try: feat.update(extract_fft_features(t, f))
    except Exception as ex: feat["_err_fft"] = str(ex)

    # ── 4.7  Autocorrelation features ─────────────────────────────────────
    try: feat.update(extract_autocorrelation_features(f))
    except Exception as ex: feat["_err_acf"] = str(ex)

    # ── 4.8  Entropy features ─────────────────────────────────────────────
    try: feat.update(extract_entropy_features(f))
    except Exception as ex: feat["_err_ent"] = str(ex)

    # ── 4.9  Hjorth parameters ────────────────────────────────────────────
    try: feat.update(extract_hjorth_features(f))
    except Exception as ex: feat["_err_hjorth"] = str(ex)

    # ── 4.10 Wavelet features ─────────────────────────────────────────────
    try: feat.update(extract_wavelet_features(f))
    except Exception as ex: feat["_err_wav"] = str(ex)

    # ── 4.11 Lomb-Scargle features ────────────────────────────────────────
    ls_period = None
    try:
        ls_feat = extract_lombscargle_features(t, f)
        ls_period = ls_feat.get("ls_best_period", None)
        feat.update(ls_feat)
    except Exception as ex:
        feat["_err_ls"] = str(ex)

    # ── 4.12 Transit / dip features ───────────────────────────────────────
    try: feat.update(extract_transit_features(t, f))
    except Exception as ex: feat["_err_transit"] = str(ex)

    # ── 4.13 Rolling window features ──────────────────────────────────────
    try: feat.update(extract_rolling_features(f))
    except Exception as ex: feat["_err_roll"] = str(ex)

    # ── 4.14 Nonlinear features ───────────────────────────────────────────
    try: feat.update(extract_nonlinear_features(f))
    except Exception as ex: feat["_err_nonlin"] = str(ex)

    # ── 4.15 catch22 features ─────────────────────────────────────────────
    try: feat.update(extract_catch22_features(f))
    except Exception as ex: feat["_err_c22"] = str(ex)

    # ── 4.16 Flux-error cross features ────────────────────────────────────
    try: feat.update(extract_flux_error_cross_features(f, e))
    except Exception as ex: feat["_err_cross"] = str(ex)

    # ── 4.17 Phase-folding features ───────────────────────────────────────
    try: feat.update(extract_phase_folding_features(t, f, ls_period=ls_period))
    except Exception as ex: feat["_err_phase"] = str(ex)

    # ── 4.18 Variability index features ──────────────────────────────────
    try: feat.update(extract_variability_index_features(f, e))
    except Exception as ex: feat["_err_varidx"] = str(ex)

    # ── 4.19 Structure function features ─────────────────────────────────
    try: feat.update(extract_structure_function_features(t, f))
    except Exception as ex: feat["_err_sf"] = str(ex)

    # ── 4.20 Flare features ───────────────────────────────────────────────
    try: feat.update(extract_flare_features(t, f, e))
    except Exception as ex: feat["_err_flare"] = str(ex)

    # ── 4.21 Envelope (Hilbert) features ─────────────────────────────────
    try: feat.update(extract_envelope_features(f))
    except Exception as ex: feat["_err_env"] = str(ex)

    # ── 4.22 Segment / CDPP features ─────────────────────────────────────
    try: feat.update(extract_segment_features(t, f))
    except Exception as ex: feat["_err_seg"] = str(ex)

    # ── 4.23 tsfresh features ─────────────────────────────────────────────
    try: feat.update(extract_tsfresh_features_single(f, target_id))
    except Exception as ex: feat["_err_tsf"] = str(ex)

    # ── 4.24 Welch PSD features ───────────────────────────────────────────
    try: feat.update(extract_welch_features(t, f))
    except Exception as ex: feat["_err_welch"] = str(ex)

    # ── 4.25 BLS proxy features ───────────────────────────────────────────
    try: feat.update(extract_bls_proxy_features(t, f))
    except Exception as ex: feat["_err_bls"] = str(ex)

    # ── 4.26 Level-crossing features ─────────────────────────────────────
    try: feat.update(extract_crossing_features(f))
    except Exception as ex: feat["_err_crossing"] = str(ex)

    # ── 4.27 Quantile-bin distribution features ───────────────────────────
    try: feat.update(extract_quantile_bin_features(f))
    except Exception as ex: feat["_err_qbin"] = str(ex)

    # ── 4.28 Multi-scale variability features ────────────────────────────
    try: feat.update(extract_multiscale_variability(f))
    except Exception as ex: feat["_err_msvar"] = str(ex)

    return feat

print("Master extractor (v2 — all 28 modules) defined.")


### Run feature extraction on all light curves

In [ ]:
# ── Smoke test on one row ─────────────────────────────────────────────────────
print("Smoke-testing master extractor on row 0 ...")
t0 = _time.time()
test_row = df.iloc[0]
test_feat = extract_all_features(
    test_row["target_id"], test_row["sector"],
    test_row["time"], test_row["flux"], test_row["flux_err"]
)
elapsed = _time.time() - t0

# Check for error keys
err_keys = [k for k in test_feat if k.startswith("_err")]
if err_keys:
    print(f"  ⚠ Errors in {len(err_keys)} modules: {err_keys}")
    for k in err_keys:
        print(f"     {k}: {test_feat[k]}")
else:
    print("  All modules ran without errors.")

n_feat_test = len([k for k in test_feat if not k.startswith("_")])
print(f"  Features extracted  : {n_feat_test}")
print(f"  Time for one row    : {elapsed:.2f}s")
print(f"  Estimated total time: {elapsed * len(df) / 60:.1f} minutes")

In [ ]:
# ── Full extraction with progress bar ────────────────────────────────────────
print(f"Extracting features from {len(df):,} light curves ...\n")

all_features = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Feature extraction"):
    try:
        feat = extract_all_features(
            row["target_id"], row["sector"],
            row["time"], row["flux"], row["flux_err"]
        )
    except Exception as ex:
        feat = {"target_id": int(row["target_id"]), "sector": int(row["sector"]),
                "_global_error": str(ex)}
    all_features.append(feat)

    # Free memory periodically
    if len(all_features) % 200 == 0:
        gc.collect()

print("\nBuilding feature DataFrame ...")
df_feat = pd.DataFrame(all_features)

# Cast to float where possible (skip id columns and error columns)
id_cols  = ["target_id", "sector"]
err_cols = [c for c in df_feat.columns if c.startswith("_")]
num_cols = [c for c in df_feat.columns if c not in id_cols + err_cols]

for col in num_cols:
    df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

print(f"Feature DataFrame shape: {df_feat.shape}")
print(f"Rows with global errors: {df_feat['_global_error'].notna().sum() if '_global_error' in df_feat.columns else 0}")
gc.collect()

## 5. Feature Selection — Remove Duplicates and Constants

In [ ]:
# ── Drop error / diagnostic columns ───────────────────────────────────────────
err_cols_present = [c for c in df_feat.columns if c.startswith("_")]
print(f"Dropping {len(err_cols_present)} diagnostic/error columns.")
df_clean = df_feat.drop(columns=err_cols_present, errors="ignore").copy()

# ── Drop constant features ────────────────────────────────────────────────────
feat_cols = [c for c in df_clean.columns if c not in ["target_id", "sector"]]
before = len(feat_cols)

# Standard deviation of each feature (NaN-safe)
stds = df_clean[feat_cols].std(ddof=1)
constant_cols = stds[stds == 0].index.tolist()
# Also drop features that are constant after ignoring NaN (all NaN case)
all_nan_cols = [c for c in feat_cols if df_clean[c].isna().all()]

to_drop = list(set(constant_cols) | set(all_nan_cols))
df_clean = df_clean.drop(columns=to_drop, errors="ignore")

print(f"  Constant features removed : {len(constant_cols)}")
print(f"  All-NaN  features removed : {len(all_nan_cols)}")

# ── Drop near-duplicate features (correlation > 0.999) ───────────────────────
feat_cols = [c for c in df_clean.columns if c not in ["target_id", "sector"]]

# Fill NaN with median for correlation computation
df_filled = df_clean[feat_cols].fillna(df_clean[feat_cols].median())

# Compute correlation matrix
print("\nComputing feature correlation matrix for deduplication...")
corr_matrix = df_filled.corr().abs()

# Upper triangle mask
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_thresh = 0.999
dup_features = [col for col in upper.columns if any(upper[col] > corr_thresh)]

df_clean = df_clean.drop(columns=dup_features, errors="ignore")
print(f"  Near-duplicate features removed (|r| > {corr_thresh}): {len(dup_features)}")

# ── Summary ───────────────────────────────────────────────────────────────────
feat_cols_final = [c for c in df_clean.columns if c not in ["target_id", "sector"]]
after = len(feat_cols_final)
print(f"\nFeatures before selection : {before}")
print(f"Features after  selection : {after}")
print(f"Features retained         : {after}")
print(f"\nFinal DataFrame shape     : {df_clean.shape}")

# NaN summary
nan_pct = df_clean[feat_cols_final].isna().mean() * 100
print(f"\nFeatures with >50% NaN    : {(nan_pct > 50).sum()}")
print(f"Features with 0%  NaN    : {(nan_pct == 0).sum()}")

In [ ]:
# ── Reorder columns: identifiers first ───────────────────────────────────────
id_cols  = ["target_id", "sector"]
other_cols = [c for c in df_clean.columns if c not in id_cols]
df_clean = df_clean[id_cols + sorted(other_cols)]

# Quick sample
print("Sample of engineered features:")
display(df_clean.iloc[:3, :10])

## 6. Export

In [ ]:
# ── Feature dictionary definition ────────────────────────────────────────────
# Maps prefix patterns to descriptions and reasons

FEATURE_CATALOGUE = [
    # (prefix, description_template, reason)
    ("target_id",         "TESS target identifier",
     "Unique identifier; required for joining back to source data."),
    ("sector",            "TESS observation sector number",
     "Provides context for time baseline and satellite orientation."),

    # Observation
    ("obs_count",         "Total number of photometric observations",
     "Longer baselines enable detection of longer-period signals."),
    ("obs_duration",      "Total observation time span in days",
     "Determines the minimum detectable frequency in periodograms."),
    ("obs_cadence",       "Time-step statistics (median, mean, std, min, max, IQR)",
     "Irregular cadence affects period recovery and noise structure."),
    ("obs_n_gaps",        "Number of observation gaps (> 2× median cadence)",
     "Gaps indicate data loss; can introduce spectral leakage."),
    ("obs_max_gap",       "Largest single observation gap",
     "Very large gaps may separate physically distinct epochs."),
    ("obs_gap_fraction",  "Fraction of total duration lost to gaps",
     "Quantifies overall data completeness."),
    ("obs_sampling_density", "Points per unit time",
     "High density improves noise averaging and short-period detection."),
    ("obs_regularity",    "Coefficient of variation of cadence (std/mean)",
     "Near-zero implies uniform sampling; high values flag irregular cadence."),

    # Statistical
    ("stat_mean",         "Mean flux",
     "First moment; baseline brightness level."),
    ("stat_median",       "Median flux",
     "Robust central location; less sensitive to outlier transits."),
    ("stat_variance",     "Variance of flux",
     "Second moment; overall variability amplitude squared."),
    ("stat_std",          "Standard deviation of flux",
     "Primary measure of flux variability amplitude."),
    ("stat_rms",          "Root mean square flux",
     "Energy-based amplitude measure; distinguishes offset signals."),
    ("stat_min",          "Minimum flux value",
     "Captures deepest dip; useful for eclipse/transit detection."),
    ("stat_max",          "Maximum flux value",
     "Captures brightest flare; useful for flare detection."),
    ("stat_range",        "Flux range (max - min)",
     "Total dynamic range of the light curve."),
    ("stat_cv",           "Coefficient of variation (std / mean)",
     "Scale-independent variability measure."),
    ("stat_skewness",     "Skewness of flux distribution",
     "Asymmetric distributions can indicate transits (negative) or flares (positive)."),
    ("stat_kurtosis",     "Excess kurtosis (Fisher definition)",
     "Heavy tails indicate rare large events; thin tails indicate bounded noise."),
    ("stat_hyper",        "5th and 6th standardised moments",
     "Higher-order shape descriptors for non-Gaussian flux distributions."),

    # Robust
    ("robust_mad",        "Median absolute deviation",
     "Robust scale estimate; not influenced by outlier dips/flares."),
    ("robust_iqr",        "Interquartile range",
     "Robust spread; insensitive to top and bottom 25% of flux values."),
    ("robust_p",          "Percentile values (5, 10, 25, 75, 90, 95)",
     "Non-parametric description of flux distribution shape."),
    ("robust_trimmed_mean","10%-trimmed mean flux",
     "Robust mean that excludes extreme 10% on each tail."),
    ("robust_winsorized_std", "Winsorized (5–95%) standard deviation",
     "Robust scale that limits influence of extreme outliers."),

    # Normalised
    ("norm_std",          "Standard deviation of median-normalised flux",
     "Scale-free variability; natural unit for transit depth estimation."),
    ("norm_range",        "Range of median-normalised flux",
     "Scale-free dynamic range."),

    # Noise
    ("noise_snr",         "Signal-to-noise ratio (std_flux / mean_flux_err)",
     "Overall photometric quality; high SNR enables detection of shallow features."),
    ("noise_err",         "Statistics of flux_err array (mean, std, median)",
     "Characterises measurement uncertainty; heteroscedastic errors need special treatment."),
    ("noise_variance",    "Mean squared flux error",
     "Average noise variance; used in chi-squared and F_var calculations."),
    ("noise_chi2_reduced","Reduced chi-squared against flat (mean) model",
     "Values >>1 indicate genuine variability beyond noise."),

    # Shape factors
    ("shape_crest_factor","Peak / RMS",
     "Measures peakedness; high values suggest impulsive events (flares)."),
    ("shape_shape_factor","RMS / mean(|f|)",
     "Shape descriptor for waveform; sensitive to distribution shape."),
    ("shape_impulse_factor","Peak / mean(|f|)",
     "Impulsiveness ratio; elevated by isolated flare events."),
    ("shape_clearance_factor","Peak / mean(sqrt(|f|))^2",
     "More sensitive to extreme peaks than crest factor."),

    # Trend
    ("trend_lin",         "Linear trend: slope, intercept, R², residual stats",
     "Long-term instrumental drift or astrophysical secular change."),
    ("trend_quad",        "Quadratic trend: curvature, slope, R², residual std",
     "Detects curved trends; curvature may indicate proximity to stellar minimum/maximum."),
    ("trend_r2_improvement", "R² gain from linear to quadratic fit",
     "Significant improvement indicates non-monotonic long-term variation."),
    ("trend_spearman_r",  "Spearman rank correlation of flux with time",
     "Monotonic trend strength; robust to non-Gaussian deviations."),
    ("trend_half_diff",   "Mean flux second half minus first half",
     "Simple brightness change across the observation; detects step-changes."),

    # Derivatives
    ("deriv1_",           "First time-derivative statistics (mean, std, max, rms, skew, kurt, MAD)",
     "Rate-of-change features; high values indicate rapid flux variations (flares, occultations)."),
    ("deriv2_",           "Second time-derivative statistics",
     "Curvature of flux profile; sensitive to sharp ingress/egress features."),
    ("deriv_zero_crossings", "Zero-crossing count of 1st and 2nd derivatives",
     "Counts local extrema; proxy for oscillation frequency."),

    # Peaks
    ("peak_",             "Peak detection: count, density, height, prominence, width, spacing, energy",
     "Characterises flux maxima (flares, pulsation maxima)."),
    ("valley_",           "Valley detection: same statistics as peaks but for minima",
     "Characterises flux minima (transits, eclipses, spot modulations)."),
    ("peak_valley",       "Combined extrema count, density, and ratio",
     "Overall oscillation structure."),

    # FFT
    ("fft_dom",           "Dominant FFT frequency, period, and power fraction",
     "Primary periodic signal if one exists."),
    ("fft_top",           "Top-10 FFT frequencies and powers",
     "Captures multiperiodic or harmonic structures."),
    ("fft_spectral_entropy", "Spectral entropy (power distribution uniformity)",
     "Low entropy = dominated by few frequencies (periodic); high = broadband noise."),
    ("fft_spectral_centroid", "Spectral centroid (power-weighted mean frequency)",
     "Average frequency of variability power."),
    ("fft_spectral_bandwidth", "Spectral bandwidth (power-weighted std of frequency)",
     "Width of spectral peak; narrow = coherent oscillation."),
    ("fft_spectral_rolloff", "Frequency below which 85% of spectral energy lies",
     "Characterises spectral shape without assuming a model."),
    ("fft_spectral_flatness", "Spectral flatness (geometric/arithmetic mean of power)",
     "Close to 1 = noise-like; close to 0 = tonal/periodic."),
    ("fft_low_high_power_ratio", "Ratio of low-frequency to high-frequency power",
     "High ratio = variability on long timescales (rotation, activity cycles)."),

    # ACF/PACF
    ("acf_lag",           "ACF values at lags 1, 2, 5, 10, 20, 50",
     "Serial correlation at specific timescales."),
    ("acf_decorrelation_lag", "First lag at which |ACF| < 1/e",
     "Coherence timescale; rotation periods often match ACF peak."),
    ("acf_first_peak",    "ACF first peak lag and value",
     "Rotation period proxy via ACF."),
    ("pacf_lag",          "PACF values at lags 1, 2, 5, 10",
     "Partial serial correlation; diagnoses autoregressive order."),

    # Entropy
    ("ent_shannon",       "Shannon entropy of flux histogram",
     "Low entropy = flux concentrated in narrow range; high = broad distribution."),
    ("ent_sample",        "Sample entropy (regularity measure)",
     "Quantifies signal complexity; high for irregular/chaotic variability."),
    ("ent_approximate",   "Approximate entropy",
     "Similar to sample entropy but biased; useful for short time series."),
    ("ent_permutation",   "Permutation entropy (ordinal pattern distribution)",
     "Detects temporal ordering complexity; robust to noise."),
    ("ent_spectral",      "Spectral entropy via Welch PSD",
     "Power-spectrum-based complexity measure."),
    ("ent_svd",           "SVD entropy of delay-embedded matrix",
     "Phase-space dimensionality proxy."),
    ("ent_lziv",          "Lempel-Ziv complexity (binarised signal)",
     "Algorithmic complexity; high for random-like, low for structured signals."),

    # Hjorth
    ("hjorth_activity",   "Hjorth Activity (variance)",
     "Signal power; equivalent to variance."),
    ("hjorth_mobility",   "Hjorth Mobility (normalised std of 1st derivative)",
     "Mean frequency of signal; proxy for dominant oscillation speed."),
    ("hjorth_complexity", "Hjorth Complexity (mobility of derivative / mobility)",
     "Deviation from a pure sine wave; indicates waveform complexity."),

    # Wavelet
    ("wav_approx",        "Approximation coefficient statistics at deepest wavelet level",
     "Low-frequency content; long-term trend and slow modulations."),
    ("wav_d",             "Detail coefficient statistics at each decomposition level (1–5)",
     "Band-pass filtered views at timescales 2^level × cadence."),
    ("wav_energy_ratio",  "Fraction of total signal energy at each wavelet level",
     "Multi-scale energy distribution; distinguishes periodic vs stochastic variability."),

    # Lomb-Scargle
    ("ls_best",           "Lomb-Scargle best frequency, period, and power",
     "Optimal period for unevenly sampled data; handles TESS data gaps correctly."),
    ("ls_top",            "Top-5 LS period peaks",
     "Multiple period candidates; aliases and harmonics."),
    ("ls_power",          "LS power statistics (mean, std, max)",
     "Describes overall periodogram shape."),
    ("ls_peak_ratio_1_2", "Ratio of strongest to second-strongest LS peak",
     "High ratio indicates clear dominant period; low ratio suggests noise."),
    ("ls_fap",            "False alarm probability at best LS power",
     "Statistical significance of the best period."),

    # Transit
    ("transit_n_dips",    "Number of significant flux dips",
     "Planet transit candidates or eclipsing binary events."),
    ("transit_deepest_dip","Deepest dip depth (relative to median flux)",
     "Proxy for transit depth; encodes planet-to-star radius ratio."),
    ("transit_mean_dip_depth", "Mean depth of all detected dips",
     "Average transit/eclipse depth."),
    ("transit_dip_depth_std", "Standard deviation of dip depths",
     "High std suggests variable dips (grazing eclipses, starspots)."),
    ("transit_mean_dip_duration", "Mean dip duration in days",
     "Encodes stellar density via transit duration formula."),
    ("transit_ingress_slope", "Mean ingress slope",
     "Steeper ingress → smaller companion; gradual → grazing geometry."),
    ("transit_egress_slope",  "Mean egress slope",
     "Steeper egress → cleaner exit from stellar limb."),
    ("transit_symmetry",  "Ingress / egress symmetry (1 = symmetric)",
     "Asymmetric transits suggest debris discs, TTVs, or non-circular orbits."),
    ("transit_period_estimate", "Estimated orbital period from dip spacing",
     "Median time between dips; independent estimate of orbital period."),
    ("transit_dip_energy","Sum of squared dip depths",
     "Total power in transit signal."),

    # Rolling
    ("roll16_",           "Rolling-window statistics (window=16)",
     "Local variability at short timescale (~2 min for TESS 2-min cadence)."),
    ("roll32_",           "Rolling-window statistics (window=32)",
     "Local variability at medium-short timescale."),
    ("roll64_",           "Rolling-window statistics (window=64)",
     "Local variability at medium timescale."),
    ("roll128_",          "Rolling-window statistics (window=128)",
     "Local variability at longer timescale."),

    # Nonlinear
    ("nonlin_hurst",      "Hurst exponent (rescaled range analysis)",
     "H > 0.5: persistent trends; H < 0.5: anti-persistent; H ≈ 0.5: random walk."),
    ("nonlin_dfa",        "Detrended fluctuation analysis exponent",
     "Scale-free correlation exponent; distinguishes fractal from periodic signals."),
    ("nonlin_lyapunov",   "Largest Lyapunov exponent",
     "Positive: chaotic; near zero: periodic; negative: stable."),
    ("nonlin_corr_dim",   "Correlation dimension",
     "Fractal dimension of attractor; low = low-dimensional chaos."),

    # catch22
    ("c22_",              "catch22 canonical time-series features",
     "22 features selected for maximum class-discriminative power across 93 datasets."),

    # Cross
    ("cross_flux_err_pearsonr", "Pearson correlation of flux vs flux_err",
     "Positive correlation may indicate Poisson noise (brighter = noisier)."),
    ("cross_weighted_mean_flux", "Error-weighted mean flux",
     "More precise mean estimate using individual measurement uncertainties."),
    ("cross_outlier_frac", "Fraction of flux points > 3σ and > 5σ from weighted mean",
     "Outlier prevalence; high fractions may indicate flares or contamination."),
    ("cross_mean_norm_err", "Mean (flux_err / |flux|)",
     "Fractional photometric precision; directly comparable across brightness levels."),

    # Phase-folding
    ("phase_binned",      "Phase-folded light curve statistics (std, range, skew, kurt)",
     "After folding on LS period; captures coherent periodic modulation shape."),
    ("phase_depth_ratio", "Relative dip depth in phase-folded curve",
     "Transit depth after phase-folding; improved SNR over single-transit depth."),

    # Variability indices
    ("var_stetson_I",     "Stetson I variability index",
     "Correlated consecutive-pair statistic; robust against noise."),
    ("var_stetson_J",     "Stetson J variability index",
     "Alternative paired-observation variability; sensitive to transient events."),
    ("var_von_neumann_eta", "Von Neumann η (adjacent-point scatter ratio)",
     "Low η = smooth variation; high η = noisy/random signal."),
    ("var_roms",          "Robust median statistic (RoMS)",
     "Robust measure of scatter relative to individual measurement errors."),
    ("var_f_var",         "Normalized excess variance F_var",
     "Intrinsic fractional variability after subtracting noise contribution."),
    ("var_mad_index",     "MAD-based flux variability index",
     "Scale-independent robust variability; less sensitive to outliers than std/mean."),

    # Structure function
    ("sf_slope",          "Log-log slope of structure function vs time lag",
     "Slope ≈ 0: white noise; ≈ 1: flicker noise; ≈ 2: random walk / sinusoidal."),
    ("sf_r2",             "R² of structure function power-law fit",
     "Goodness-of-fit of the power-law model."),
    ("sf_turnover_tau",   "Time lag at maximum structure function value",
     "Characteristic timescale at which variability saturates."),
]

# Build feature dictionary DataFrame
feat_dict_rows = []
final_feat_cols = set(df_clean.columns)

for col in sorted(df_clean.columns):
    if col in ["target_id", "sector"
    # ── Flare features (4.20) ────────────────────────────────────────────────
    ("flare_count",           "Number of detected flux excursions > 3.5σ above median",
     "Direct count of stellar flare events; key discriminator for active M-dwarfs."),
    ("flare_total_energy",    "Integrated energy of all flares (flux × time integral)",
     "Total flare energy budget; correlated with stellar activity level and age."),
    ("flare_mean_amplitude",  "Mean flare peak amplitude above quiescent level",
     "Characterises typical flare intensity; related to magnetic field strength."),
    ("flare_max_amplitude",   "Largest single flare amplitude",
     "Extreme flare size; important for habitability studies."),
    ("flare_mean_duration",   "Mean flare duration in days",
     "Short flares suggest impulsive events; long flares suggest complex events."),
    ("flare_rate",            "Flares per day (flare frequency)",
     "Activity rate; higher in younger/more active stars."),
    ("flare_energy_frac",     "Fraction of total flux energy in flares",
     "Relative flare contribution to the total luminosity budget."),

    # ── Envelope / Hilbert features (4.21) ───────────────────────────────────
    ("env_mean",              "Mean of analytic signal amplitude envelope",
     "Average oscillation amplitude; related to variability amplitude."),
    ("env_std",               "Standard deviation of envelope",
     "How much the oscillation amplitude varies over time."),
    ("env_cv",                "Coefficient of variation of envelope",
     "Scale-independent amplitude modulation strength."),
    ("env_am_depth",          "Amplitude modulation depth (max-min)/(max+min)",
     "Measures whether variability is modulated (e.g. beating between close periods)."),
    ("inst_freq_mean",        "Mean instantaneous frequency",
     "Average rate of phase change; proxy for dominant frequency."),
    ("inst_freq_std",         "Std of instantaneous frequency",
     "High std = frequency-modulated or non-stationary oscillation."),

    # ── Segment / CDPP features (4.22) ───────────────────────────────────────
    ("seg_mean_mean",         "Mean of per-segment flux means",
     "Overall average brightness across temporal segments."),
    ("seg_mean_range",        "Range of per-segment flux means",
     "Long-term brightness variation; large range = secular variability."),
    ("seg_std_mean",          "Mean of per-segment standard deviations",
     "Average local variability; smoother than global std."),
    ("seg_std_range",         "Range of per-segment standard deviations",
     "Variability changes over time; large range = non-stationary variability."),
    ("seg_kruskal_H",         "Kruskal-Wallis H statistic across time segments",
     "Non-parametric test for non-stationarity; large H = varying distribution."),
    ("seg_kruskal_p",         "p-value of Kruskal-Wallis test",
     "p < 0.05 indicates statistically significant non-stationarity."),
    ("cdpp_proxy_ppm",        "Combined Differential Photometric Precision proxy (ppm, 1-hr)",
     "Standard TESS data quality metric; lower = better photometric precision."),

    # ── tsfresh features (4.23) ───────────────────────────────────────────────
    ("tsf_",                  "tsfresh EfficientFCParameters features",
     "~800 curated time-series features with proven discriminative power across many domains."),

    # ── Welch PSD features (4.24) ─────────────────────────────────────────────
    ("welch_dom_freq",        "Dominant frequency from Welch PSD",
     "More robust period estimate than plain FFT due to segment averaging."),
    ("welch_dom_period",      "Dominant period from Welch PSD",
     "Period corresponding to highest Welch power."),
    ("welch_band",            "Power in 4 log-spaced frequency bands",
     "Spectral energy distribution across timescales."),
    ("welch_spectral_slope",  "Log-log slope of Welch PSD (spectral index)",
     "Slope = -2: red noise; -1: pink noise; 0: white noise. Characterises variability type."),

    # ── BLS proxy features (4.25) ─────────────────────────────────────────────
    ("bls_best_period",       "Best Box Least Squares period",
     "Gold-standard transit search algorithm; most sensitive to box-shaped dips."),
    ("bls_best_depth",        "BLS best transit depth",
     "Fractional flux decrement; (Rp/Rs)^2 for planet transits."),
    ("bls_best_duration",     "BLS transit duration in days",
     "Duration encodes stellar density and orbital geometry."),
    ("bls_snr",               "BLS transit SNR",
     "Statistical significance of the best transit signal."),
    ("bls_odd_even_diff",     "BLS odd-even transit depth difference",
     "Non-zero values indicate eclipsing binary rather than planet transit."),

    # ── Level-crossing features (4.26) ────────────────────────────────────────
    ("cross_rate_mean",       "Mean-crossing rate (zero-crossings per sample of demeaned flux)",
     "Frequency proxy; high rate implies rapid oscillations."),
    ("cross_rate_median",     "Median-crossing rate",
     "Similar to mean-crossing rate but robust to asymmetric distributions."),
    ("cross_frac_above_p",    "Fraction of observations above various flux percentiles",
     "Non-parametric duty cycle; captures flux distribution asymmetry."),
    ("cross_run_mean",        "Mean run length above/below median",
     "Average duration of excursions; long runs suggest slow modulation."),
    ("cross_run_max",         "Maximum run length above/below median",
     "Longest persistent excursion; may indicate rotation modulation or eclipse."),

    # ── Quantile-bin features (4.27) ──────────────────────────────────────────
    ("qbin_",                 "Fraction of observations in each flux quantile bin",
     "Non-parametric flux distribution fingerprint; sensitive to bimodality."),
    ("qbin_l1_from_uniform",  "L1 distance of bin fractions from uniform distribution",
     "High value = concentration in few flux states; transit/pulsation signature."),
    ("qbin_entropy",          "Entropy of quantile bin distribution",
     "Low entropy = flux concentrated in few states (e.g. eclipse flat bottom)."),

    # ── Multi-scale variability features (4.28) ───────────────────────────────
    ("msvar_rms_scale",       "RMS of flux differences at various time strides",
     "Allan deviation-like measure; reveals fractal/self-similar variability structure."),
    ("msvar_ratio",           "Ratio of RMS variability between consecutive scales",
     "Close to 1 = scale-invariant (fractal); deviations indicate characteristic timescale."),
    ("msvar_slope",           "Log-log slope of multi-scale RMS vs scale",
     "= H - 0.5 for fractional Brownian motion; H>0.5 = persistent, H<0.5 = anti-persistent."),
]:
        continue

    # Find matching entry in catalogue
    matched_desc   = "Feature extracted from TESS light curve"
    matched_reason = "Contributes information about the variability structure."

    for prefix, desc, reason in FEATURE_CATALOGUE:
        if col.startswith(prefix) or prefix in col:
            matched_desc   = desc
            matched_reason = reason
            break

    feat_dict_rows.append({
        "feature_name"       : col,
        "description"        : matched_desc,
        "reason_for_including": matched_reason,
    })

df_feature_dict = pd.DataFrame(feat_dict_rows)
print(f"Feature dictionary: {len(df_feature_dict)} entries")
display(df_feature_dict.head(10))

In [ ]:
# ── Write outputs ─────────────────────────────────────────────────────────────
OUT_CSV      = "engineered_features.csv"
OUT_PARQUET  = "engineered_features.parquet"
OUT_DICT_CSV = "feature_dictionary.csv"

print("Saving files ...")

df_clean.to_csv(OUT_CSV, index=False)
print(f"  {OUT_CSV}  ({os.path.getsize(OUT_CSV)/1e6:.1f} MB)")

df_clean.to_parquet(OUT_PARQUET, index=False)
print(f"  {OUT_PARQUET}  ({os.path.getsize(OUT_PARQUET)/1e6:.1f} MB)")

df_feature_dict.to_csv(OUT_DICT_CSV, index=False)
print(f"  {OUT_DICT_CSV}  ({os.path.getsize(OUT_DICT_CSV)/1e6:.2f} MB)")

print("\nAll files saved successfully.")

## 7. Final Report

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FINAL REPORT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

feat_cols_final = [c for c in df_clean.columns if c not in ["target_id", "sector"]]

# ── Feature category breakdown ────────────────────────────────────────────────
CATEGORIES = {
    "Observation"          : lambda c: c.startswith("obs_"),
    "Statistical (std)"    : lambda c: c.startswith("stat_"),
    "Robust statistics"    : lambda c: c.startswith("robust_"),
    "Normalised flux"      : lambda c: c.startswith("norm_"),
    "Noise / SNR"          : lambda c: c.startswith("noise_"),
    "Shape factors"        : lambda c: c.startswith("shape_"),
    "Trend"                : lambda c: c.startswith("trend_"),
    "Derivatives"          : lambda c: c.startswith("deriv"),
    "Peaks / Valleys"      : lambda c: c.startswith("peak_") or c.startswith("valley_"),
    "FFT / Spectral"       : lambda c: c.startswith("fft_"),
    "Autocorrelation"      : lambda c: c.startswith("acf_") or c.startswith("pacf_"),
    "Entropy"              : lambda c: c.startswith("ent_"),
    "Hjorth"               : lambda c: c.startswith("hjorth_"),
    "Wavelet"              : lambda c: c.startswith("wav_"),
    "Lomb-Scargle"         : lambda c: c.startswith("ls_"),
    "Transit / Dip"        : lambda c: c.startswith("transit_"),
    "Rolling window"       : lambda c: c.startswith("roll"),
    "Nonlinear dynamics"   : lambda c: c.startswith("nonlin_"),
    "catch22"              : lambda c: c.startswith("c22_"),
    "Flux-error cross"     : lambda c: c.startswith("cross_"),
    "Phase-folding"        : lambda c: c.startswith("phase_"),
    "Variability indices"  : lambda c: c.startswith("var_"),
    "Structure function"   : lambda c: c.startswith("sf_"),
}

print("="*70)
print("            TESS FEATURE ENGINEERING — FINAL REPORT")
print("="*70)

print(f"\n  Dataset       : {DATA_PATH}")
print(f"  Light curves  : {len(df_clean):,}")
print(f"  Total features: {len(feat_cols_final)}")

print("\n── Feature Category Breakdown ─────────────────────────────────────")
total_categorised = 0
for cat_name, cat_fn in CATEGORIES.items():
    n = sum(1 for c in feat_cols_final if cat_fn(c))
    total_categorised += n
    bar = "█" * (n // 2)
    print(f"  {cat_name:<25s}: {n:4d}  {bar}")

uncategorised = len(feat_cols_final) - total_categorised
if uncategorised > 0:
    print(f"  {'Other / Mixed':<25s}: {uncategorised:4d}")

print("\n── NaN Summary ────────────────────────────────────────────────────")
nan_pct_all = df_clean[feat_cols_final].isna().mean() * 100
print(f"  Features with 0% NaN     : {(nan_pct_all == 0).sum()}")
print(f"  Features with <10% NaN   : {(nan_pct_all < 10).sum()}")
print(f"  Features with >50% NaN   : {(nan_pct_all > 50).sum()}")
print(f"  Mean NaN rate across features: {nan_pct_all.mean():.1f}%")

print("\n── Top 10 Most Variable Features (highest coefficient of variation) ─")
feat_cv = (df_clean[feat_cols_final].std() / (df_clean[feat_cols_final].mean().abs() + 1e-30)).abs()
for feat, val in feat_cv.nlargest(10).items():
    print(f"  {feat:<45s}: CV = {val:.3f}")

print("\n── Interesting Observations ────────────────────────────────────────")

# Observation duration spread
if "obs_duration" in df_clean.columns:
    dur_min = df_clean["obs_duration"].min()
    dur_max = df_clean["obs_duration"].max()
    print(f"  Observation duration range: {dur_min:.1f} – {dur_max:.1f} days")

# SNR distribution
if "noise_snr" in df_clean.columns:
    snr_med = df_clean["noise_snr"].median()
    snr_p10 = df_clean["noise_snr"].quantile(0.10)
    snr_p90 = df_clean["noise_snr"].quantile(0.90)
    print(f"  SNR: median={snr_med:.1f}, P10={snr_p10:.1f}, P90={snr_p90:.1f}")

# Fraction with significant dips
if "transit_n_dips" in df_clean.columns:
    frac_dips = (df_clean["transit_n_dips"] > 0).mean() * 100
    print(f"  Light curves with ≥1 significant dip: {frac_dips:.1f}%")

# Lomb-Scargle FAP
if "ls_fap" in df_clean.columns:
    frac_periodic = (df_clean["ls_fap"] < 0.01).mean() * 100
    print(f"  Light curves with LS FAP < 1%: {frac_periodic:.1f}% (likely periodic)")

# Fraction with high Stetson I
if "var_stetson_I" in df_clean.columns:
    frac_var = (df_clean["var_stetson_I"].abs() > 0.5).mean() * 100
    print(f"  Light curves with |Stetson I| > 0.5 (variable): {frac_var:.1f}%")

print("\n── Data Quality Issues ─────────────────────────────────────────────")
for issue in issues:
    print(f"  ⚠  {issue}")
for fix in fixes:
    print(f"  ✓  {fix}")
if not issues:
    print("  None identified.")

print("\n── Output Files ────────────────────────────────────────────────────")
for fn in [OUT_CSV, OUT_PARQUET, OUT_DICT_CSV]:
    sz = os.path.getsize(fn) / 1e6 if os.path.exists(fn) else 0
    print(f"  {fn:<35s}: {sz:.2f} MB")

print("\n" + "="*70)
print("Feature engineering complete.")
print("="*70)

### 7b. Feature Distribution Visualisation

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7b. Feature Distribution Visualisation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

feat_cols_final = [c for c in df_clean.columns if c not in ["target_id","sector"]]

# ── 1. NaN heatmap by feature category ────────────────────────────────────────
CATEGORIES = {
    "obs": "obs_", "stat": "stat_", "robust": "robust_",
    "noise": "noise_", "shape": "shape_", "trend": "trend_",
    "deriv": "deriv", "peak": "peak_", "fft": "fft_",
    "acf": "acf_", "ent": "ent_", "hjorth": "hjorth_",
    "wav": "wav_", "ls": "ls_", "transit": "transit_",
    "roll": "roll", "nonlin": "nonlin_", "c22": "c22_",
    "cross": "cross_", "phase": "phase_", "var": "var_",
    "sf": "sf_", "flare": "flare_", "env": "env_",
    "seg": "seg_", "cdpp": "cdpp_", "tsf": "tsf_",
    "welch": "welch_", "bls": "bls_", "crossing": "cross_rate",
    "qbin": "qbin_", "msvar": "msvar_",
}

cat_nan = {}
for cat, pfx in CATEGORIES.items():
    cols = [c for c in feat_cols_final if c.startswith(pfx)]
    if cols:
        cat_nan[cat] = df_clean[cols].isna().mean().mean() * 100

if cat_nan:
    fig, ax = plt.subplots(figsize=(14, 4))
    cats   = list(cat_nan.keys())
    values = [cat_nan[c] for c in cats]
    colors = ["#d62728" if v > 50 else "#ff7f0e" if v > 10 else "#2ca02c" for v in values]
    ax.bar(cats, values, color=colors)
    ax.axhline(50, color="red",    linestyle="--", linewidth=1, label="50% NaN")
    ax.axhline(10, color="orange", linestyle="--", linewidth=1, label="10% NaN")
    ax.set_ylabel("Mean NaN % within category")
    ax.set_xlabel("Feature category")
    ax.set_title("NaN Rate by Feature Category")
    ax.legend(fontsize=8)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    plt.savefig("nan_by_category.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: nan_by_category.png")

# ── 2. Key feature distributions ──────────────────────────────────────────────
KEY_FEATS = [
    "noise_snr", "stat_std", "ls_best_period", "transit_n_dips",
    "var_stetson_I", "ent_permutation", "fft_spectral_entropy",
    "bls_snr", "sf_slope", "hjorth_complexity",
]

available_key = [c for c in KEY_FEATS if c in df_clean.columns]
n_plot = len(available_key)
if n_plot > 0:
    ncols = 5
    nrows = (n_plot + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.5 * nrows))
    axes = np.array(axes).flatten()

    for i, col in enumerate(available_key):
        data = df_clean[col].dropna()
        axes[i].hist(data, bins=50, color="steelblue", edgecolor="none", alpha=0.8)
        axes[i].set_title(col, fontsize=8)
        axes[i].set_xlabel("", fontsize=7)
        axes[i].tick_params(labelsize=7)
        med = data.median()
        axes[i].axvline(med, color="red", linewidth=1, linestyle="--")

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Key Feature Distributions (red = median)", fontsize=12)
    plt.tight_layout()
    plt.savefig("key_feature_distributions.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: key_feature_distributions.png")

# ── 3. Correlation heatmap of key features ────────────────────────────────────
if n_plot >= 4:
    corr_sub = df_clean[available_key].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_sub.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(available_key)))
    ax.set_yticks(range(len(available_key)))
    ax.set_xticklabels([c.replace("_"," ") for c in available_key], rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels([c.replace("_"," ") for c in available_key], fontsize=8)
    plt.colorbar(im, ax=ax, label="Pearson r")
    ax.set_title("Correlation Matrix — Key Features", fontsize=12)
    # Annotate
    for ii in range(len(available_key)):
        for jj in range(len(available_key)):
            v = corr_sub.values[ii, jj]
            ax.text(jj, ii, f"{v:.2f}", ha="center", va="center",
                    fontsize=6, color="black" if abs(v)<0.7 else "white")
    plt.tight_layout()
    plt.savefig("key_feature_correlation.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: key_feature_correlation.png")

# ── 4. CDPP vs SNR scatter ────────────────────────────────────────────────────
if "cdpp_proxy_ppm" in df_clean.columns and "noise_snr" in df_clean.columns:
    cdpp_vals = df_clean["cdpp_proxy_ppm"].clip(0, 5000)
    snr_vals  = df_clean["noise_snr"].clip(0, 500)
    mask = cdpp_vals.notna() & snr_vals.notna()
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(snr_vals[mask], cdpp_vals[mask], s=5, alpha=0.3, c="steelblue")
    ax.set_xlabel("SNR (std_flux / mean_flux_err)")
    ax.set_ylabel("CDPP proxy (ppm, 1-hr bins)")
    ax.set_title("Photometric Precision: CDPP vs SNR")
    plt.tight_layout()
    plt.savefig("cdpp_vs_snr.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: cdpp_vs_snr.png")

print("\nAll diagnostic plots saved.")
